# Practical 3 — Focal mechanisms (SKHASH) and stress inversion: the Central Marmara sequences of 2025 and 2019

By Francisca Guinez-Rivas and Sebastian Nunez Jara

Hands-on class with the data of the 2025 Mw 6.3 Central Marmara (Türkiye) earthquake sequence (Núñez-Jara et al., 2026, GFZ Data Services, https://doi.org/10.5880/GFZ.ITSJ.2026.002) for Parts 1 and 2, and the focal mechanisms of the neighboring 2019 Mw 5.8 Silivri sequence for Part 3. Three parts:

| Part | Goal | You will |
|---|---|---|
| **1** | Evaluate the seismic network | Map the stations that recorded the sequence and the mainshock epicenter |
| **2** | Focal mechanisms with **SKHASH** | See where a station plots on the focal sphere (scheme and exercise), see what ocean-bottom stations would add (amphibious network), choose the stations worth picking, pick the P polarities station by station (interactive), map your picks, run SKHASH, compare with the published solutions and the mapped faults, and see how the solution changes when you change the stations (distance, azimuthal coverage, number of stations, ocean-bottom stations) |
| **3** | Stress field (2019 Silivri sequence) | Map the mechanisms with the faults, state what you expect, run **StressInverse** step by step (what goes in, what comes out), interpret the result, check whether it is reliable and how it changes with your choices, see which earthquakes fit it, map S<sub>Hmax</sub> and S<sub>hmin</sub>, and ask which nodal plane is the fault |

The three events of Part 2 have magnitudes between 3 and 4 and a quality-A mechanism in the published catalog: the **M 3.8 foreshock 36 minutes before the mainshock** (event 2815), a strike-slip aftershock (4009) and a normal-faulting aftershock (8317).

**Data**

The data are downloaded automatically in *Prepare the workspace* from the [GitHub Release](https://github.com/sebmagia/applied_seismology_rwth/releases/tag/focal-mechanisms-stress-practical-v1) of this practical, into the folder `focal_stress_data/` next to the notebook:

```
focal_stress_data/
├── data/                        <- data.zip
│   ├── stations.csv             <- stations that recorded the sequence (without the GONAF borehole network)
│   ├── events.csv               <- the 3 events of Part 2
│   ├── picks.csv                <- PhaseNet P and S picks of these events (from the catalog)
│   ├── mseed_2025/EV<event_id>_raw.mseed   <- 120-s three-component raw waveforms of each event (e.g. EV02815_raw.mseed)
│   ├── skhash/<event_id>/       <- SKHASH input files of each event (eq_catalog.csv, pol.csv, control_file.txt); your picks go to pol.csv
│   ├── mechanisms_catalog.csv   <- published focal-mechanism catalog of 2025 (reference solutions in Part 2)
│   ├── mechanisms_catalog_2019.csv  <- focal mechanisms of the 2019 Silivri sequence (Part 3)
│   ├── faultMRM.flt             <- mapped fault traces of the Main Marmara Fault (lon lat, segments separated by '>')
│   ├── figures/                 <- figure of the focal mechanisms of the 2025 sequence (Part 2)
│   ├── make_skhash_2025.py      <- script that made the files above from the data publication (folder data_2025/)
│   └── data_2025/               <- the files of the data publication used by make_skhash_2025.py
└── StressInverse/               <- StressInverse.zip: the StressInverse package (Python); Data/Marmara_fms.dat = the 2019 mechanisms
```

All the paths of the notebook start from `DATA_ROOT` (*Prepare the workspace*), so they do not depend on the folder Jupyter is working in.

**Requirements:** all in the course environment `applied-seismology` (see `README.md`): `obspy`, `numpy`, `pandas`, `matplotlib`, `scipy`, `cartopy`, `SKHASH` and `ipywidgets` for the buttons of the interactive exercises (without it, you type the answers instead).

> **Tip:** run the cells in order. Cells that start with `# --- helper code` only define functions: run them, you do not need to read them.

## Prepare the workspace

This notebook runs in the course environment `applied-seismology` (see `README.md` in the course folder); the first cell below checks it.

The waveforms, station list, catalogs, SKHASH input files and the StressInverse package are downloaded automatically from the [GitHub Release](https://github.com/sebmagia/applied_seismology_rwth/releases/tag/focal-mechanisms-stress-practical-v1) of this practical (or in advance with `python download_data.py focal`).

**Your picks are files** (`skhash/<event_id>/pol.csv` inside `focal_stress_data/data/`). On your computer they stay there (delete the folder `focal_stress_data/` to start again from scratch).

In [ ]:
# Check that this notebook runs in the course environment (see README.md).
import sys
from importlib.util import find_spec

missing = [pkg for pkg in ('obspy', 'cartopy', 'SKHASH', 'ipywidgets', 'ipympl') if find_spec(pkg) is None]
if missing:
    raise ModuleNotFoundError(
        f"Not installed for this kernel ({sys.executable}): {', '.join(missing)}. "
        "Start Jupyter from the course environment, or select its kernel (see README.md).")
print("Course environment OK:", sys.prefix)

### Import the packages

In [ ]:
# Interactive figures (zoom, pan) with ipympl, for all the figures of this notebook.
import sys, shutil, subprocess, warnings, urllib.request, zipfile
%matplotlib widget

from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from obspy import read, Stream, UTCDateTime
from obspy.geodetics import gps2dist_azimuth
from obspy.imaging.beachball import beach
from IPython.display import display
import cartopy.crs as ccrs
import cartopy.feature as cfeature

try:
    import ipywidgets as widgets            # buttons of the interactive exercises (JupyterLab, Jupyter Notebook, VS Code)
except ImportError:
    widgets = None                          # without ipywidgets you type the answers instead

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "figure.max_open_warning": 200})   # interactive figures stay open: closing one would remove it from the notebook

# Record the versions used in this run.
print("python", sys.version.split()[0])
for pkg in ("numpy", "scipy", "pandas", "matplotlib", "obspy", "cartopy", "SKHASH", "ipywidgets"):
    try:
        print(pkg, version(pkg))
    except PackageNotFoundError:
        try:
            __import__(pkg); print(pkg, "installed")
        except ImportError:
            print(pkg, "NOT INSTALLED")

### Download the data

The next cell downloads the data of the practical (about 20 MB) the first time, into `focal_stress_data/` next to the notebook; afterwards it finds them there. The cell after it defines the paths and a few helper functions used everywhere.

In [ ]:
RELEASE_BASE = (
    "https://github.com/sebmagia/applied_seismology_rwth/"
    "releases/download/focal-mechanisms-stress-practical-v1"
)

DATA_ROOT = Path.cwd() / "focal_stress_data"   # next to this notebook
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Download and extract each archive only if its folder is not there yet.
for name in ["data", "StressInverse"]:
    if (DATA_ROOT / name).exists():
        print(f"{name}/ already available")
        continue

    zip_path = DATA_ROOT / f"{name}.zip"
    print(f"Downloading {RELEASE_BASE}/{name}.zip ...")
    urllib.request.urlretrieve(f"{RELEASE_BASE}/{name}.zip", zip_path)

    print(f"Extracting {zip_path.name} ...")
    tmp = DATA_ROOT / f".{name}_tmp"                # extract aside first: an interrupted run leaves no half folder
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp)
    (tmp / name).rename(DATA_ROOT / name)
    shutil.rmtree(tmp, ignore_errors=True)
    zip_path.unlink(missing_ok=True)

print("Dataset root:", DATA_ROOT)
for d in sorted(DATA_ROOT.iterdir()):
    print(" ", d.name)

In [ ]:
# --- helper code: paths and small functions used everywhere. Just run this cell ---
# ---------------- paths: all inside DATA_ROOT (absolute, so they work whatever the working folder is) ----------------
CLASS_DIR = DATA_ROOT/"data"
STATIONS_CSV = CLASS_DIR/"stations.csv"
EVENTS_CSV = CLASS_DIR/"events.csv"
PICKS_CSV = CLASS_DIR/"picks.csv"
WAVEFORM_DIR = CLASS_DIR/"mseed_2025"                  # EV<event_id>_raw.mseed, e.g. EV02815_raw.mseed
SKHASH_DIR = CLASS_DIR/"skhash"                        # SKHASH input files and your picks (pol.csv), one folder per event
CATALOG_CSV = CLASS_DIR/"mechanisms_catalog.csv"           # 2025 focal mechanisms (published solutions, Part 2)
CATALOG_2019_CSV = CLASS_DIR/"mechanisms_catalog_2019.csv" # 2019 Silivri sequence focal mechanisms (Part 3)
FAULTS_FILE = CLASS_DIR/"faultMRM.flt"                     # mapped traces of the Main Marmara Fault system
STRESSINVERSE_DIR = DATA_ROOT/"StressInverse"              # the StressInverse package (Part 3)
for _p in (STATIONS_CSV, EVENTS_CSV, PICKS_CSV, WAVEFORM_DIR, SKHASH_DIR, CATALOG_CSV, CATALOG_2019_CSV, FAULTS_FILE, STRESSINVERSE_DIR):
    if not _p.exists():
        print(f"WARNING: {_p} not found. Run the download cell above (or delete {DATA_ROOT} and run it again).")

# 2025 Central Marmara mainshock (event 2818 of the catalog) and 2019 Silivri mainshock (Part 3)
MAINSHOCK = dict(lat=40.8404, lon=28.2257, label="Mainshock (23 Apr 2025, M 6.3)")
MAINSHOCK_2019 = dict(lat=40.890, lon=28.173, label="Mainshock (26 Sep 2019, Mw 5.8)")
RNG = np.random.default_rng(1)

def event_file(ev_id):
    """Waveform file of an event in WAVEFORM_DIR: EV<event_id>_raw.mseed, with or without leading zeros (EV02815 = EV2815)."""
    for f in sorted(WAVEFORM_DIR.glob("EV*_raw.mseed")):
        num = f.name[2:-len("_raw.mseed")]
        if num.isdigit() and int(num) == int(ev_id):
            return f
    return WAVEFORM_DIR/f"EV{int(ev_id):05d}_raw.mseed"            # not there: the name we expected

# ---------------- small plotting helpers ----------------
_COAST = None
def add_coast(ax):
    # coastlines through cartopy's Natural Earth data (if cartopy is installed and can download them)
    global _COAST
    try:
        import cartopy.io.shapereader as shp
        from shapely.geometry import box
        if _COAST is None:
            _COAST = list(shp.Reader(shp.natural_earth("10m", "physical", "coastline")).geometries())
        xl, yl = ax.get_xlim(), ax.get_ylim(); bb = box(xl[0], yl[0], xl[1], yl[1])
        for g in _COAST:
            if g.intersects(bb):
                c = g.intersection(bb)
                for line in getattr(c, "geoms", [c]):
                    if hasattr(line, "xy"): ax.plot(*line.xy, color="0.35", lw=0.7, zorder=1)
        ax.set_xlim(xl); ax.set_ylim(yl)
    except Exception:
        pass

def read_faults(path=FAULTS_FILE):
    """Fault traces of a GMT multi-segment file (lines 'lon lat', segments separated by '>')."""
    segs, cur = [], []
    if not Path(path).exists():
        print(f"WARNING: fault file {path} not found: the maps will have no faults"); return segs
    for line in open(path):
        t = line.strip()
        if t.startswith(">"):
            if len(cur) > 1: segs.append(np.array(cur))
            cur = []
        elif t:
            cur.append([float(v) for v in t.split()[:2]])
    if len(cur) > 1: segs.append(np.array(cur))
    return list({tuple(np.round(s, 6).ravel()): s for s in segs}.values())      # the file repeats some segments

FAULTS = read_faults()

def add_faults(ax, transform=None, label="mapped faults", **kw):
    """Draw the fault traces (FAULTS) on a map. On cartopy maps give transform=ccrs.PlateCarree()."""
    from matplotlib.collections import LineCollection
    if not FAULTS: return
    sty = dict(colors="firebrick", linewidths=0.8, alpha=0.9, zorder=2.5, label=label); sty.update(kw)
    if transform is not None: sty["transform"] = transform
    ax.add_collection(LineCollection(FAULTS, **sty), autolim=False)          # do not change the map extent

def map_axes(ax, lon_range, lat_range, title=None):
    ax.set_xlim(*lon_range); ax.set_ylim(*lat_range)
    ax.set_aspect(1/np.cos(np.radians(np.mean(lat_range))))
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)"); ax.grid(alpha=0.3, lw=0.4)
    add_coast(ax)
    if title: ax.set_title(title, fontsize=10)

def offset_point(lat0, lon0, dist_km, az_deg):
    a = np.radians(az_deg); dlat = dist_km*np.cos(a)/111.19
    return lat0 + dlat, lon0 + dist_km*np.sin(a)/(111.19*np.cos(np.radians(lat0)))

def dist_az(lat0, lon0, lat1, lon1):
    d, az, _ = gps2dist_azimuth(lat0, lon0, lat1, lon1); return d/1000.0, az

def azimuthal_gap(az):
    # largest angle (deg) between two consecutive station azimuths seen from the epicenter
    a = np.sort(np.asarray(az, float) % 360)
    return 360.0 if len(a) < 2 else np.diff(np.r_[a, a[0] + 360]).max()

def fault_vectors(strike, dip, rake):
    """Unit fault normal n and slip vector u of a mechanism (x = north, y = east, z = down; Aki & Richards)."""
    s, d, r = np.radians([strike, dip, rake])
    n = np.array([-np.sin(d)*np.sin(s), np.sin(d)*np.cos(s), -np.cos(d)])
    u = np.array([np.cos(r)*np.cos(s) + np.cos(d)*np.sin(r)*np.sin(s), np.cos(r)*np.sin(s) - np.cos(d)*np.sin(r)*np.cos(s), -np.sin(r)*np.sin(d)])
    return n, u

def kagan_angle(sdr1, sdr2):
    """Smallest rotation angle (deg) between two double-couple mechanisms given as (strike, dip, rake)."""
    def axes(sdr):
        n, u = fault_vectors(*sdr); t, p = (n + u)/np.sqrt(2), (n - u)/np.sqrt(2)
        return np.column_stack([t, p, np.cross(t, p)])                            # T, P, B axes
    a1, a2 = axes(sdr1), axes(sdr2)
    sym = [np.diag(v) for v in ([1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1])]   # T, P, B are defined up to sign
    return min(np.degrees(np.arccos(np.clip((np.trace(a1 @ s @ a2.T) - 1)/2, -1, 1))) for s in sym)

def star(ax, lat, lon, label=None, size=380, **kw):
    ax.scatter(lon, lat, marker="*", s=size, c="gold", ec="k", lw=1.0, zorder=10, label=label, **kw)

## Part 1 — Evaluate the seismic network

The file `stations.csv` lists the broadband stations that recorded the **2025 Central Marmara earthquake sequence** (networks of AFAD `TU`, KOERI `KO` and PIRES `PZ`; the GONAF borehole network was left out because its sensors behave badly). Columns: `station, location, net, latitude, longitude, elevation`. Below we map them together with the epicenter of the M 6.3 mainshock (star) at **40.840°N, 28.226°E** (23 April 2025, 09:49 UTC, depth 12 km).

In [ ]:
stations = pd.read_csv(STATIONS_CSV, skipinitialspace=True, dtype={"location": str}).rename(columns={"net": "network"})
stations["station"] = stations.station.str.strip(); stations["network"] = stations.network.str.strip()
stations[["dist_km", "az"]] = [dist_az(MAINSHOCK["lat"], MAINSHOCK["lon"], r.latitude, r.longitude) for r in stations.itertuples()]
print(f"{len(stations)} stations, {stations.network.nunique()} networks")

nets = sorted(stations.network.unique()); colors = dict(zip(nets, plt.cm.tab10(np.linspace(0, 1, 10))))
fig, ax = plt.subplots(figsize=(10, 6.2))
pad = 0.25
map_axes(ax, (stations.longitude.min()-pad, stations.longitude.max()+pad), (stations.latitude.min()-pad, stations.latitude.max()+pad))
for net in nets:
    s = stations[stations.network == net]
    ax.scatter(s.longitude, s.latitude, marker="^", s=38, color=colors[net], ec="k", lw=0.4, zorder=5, label=f"{net} ({len(s)})")
for r_km in (50, 100):                                    # distance rings around the mainshock
    la, lo = zip(*[offset_point(MAINSHOCK["lat"], MAINSHOCK["lon"], r_km, a) for a in np.linspace(0, 360, 181)])
    ax.plot(lo, la, "k--", lw=0.6, alpha=0.6, zorder=2); ax.text(lo[90], la[90]-0.04, f"{r_km} km", fontsize=7, ha="center", va="top")
add_faults(ax)                                            # mapped traces of the Main Marmara Fault
star(ax, MAINSHOCK["lat"], MAINSHOCK["lon"], MAINSHOCK["label"])
ax.legend(title="Network (n stations)", loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8)
ax.set_title("Seismic network that recorded the 2025 Central Marmara earthquake sequence", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Quick network evaluation relative to the mainshock
print(stations.groupby("network").agg(n=("station", "size"), median_dist_km=("dist_km", "median")).round(0).astype({"median_dist_km": int}).to_string(), "\n")
for r in (25, 50, 100, 150):
    print(f"stations within {r:3d} km of the mainshock: {(stations.dist_km <= r).sum()}")

for r in (50, 100, 150):
    s = stations[stations.dist_km <= r]; print(f"azimuthal gap using stations within {r:3d} km: {azimuthal_gap(s.az):5.1f}°")

fig = plt.figure(figsize=(9, 3.6))
ax0 = fig.add_subplot(1, 2, 1); axp = fig.add_subplot(1, 2, 2, projection="polar")
ax0.hist(stations.dist_km, bins=np.arange(0, stations.dist_km.max()+10, 10), color="C0", ec="k"); ax0.set_xlabel("Distance to mainshock (km)"); ax0.set_ylabel("Number of stations")
axp.set_theta_zero_location("N"); axp.set_theta_direction(-1)
axp.hist(np.radians(stations.az), bins=np.radians(np.arange(0, 361, 20)), color="C1", ec="k"); axp.set_title("Station azimuths", fontsize=9)
plt.tight_layout(); plt.show()

#### Exercise 1

- Which networks dominate and where are they? Are there azimuths with no stations? (Remember that the epicenter is under the sea.)
- How many stations are within 50 and 100 km? What is the azimuthal gap?
- A good focal mechanism needs stations at many azimuths **and** many take-off angles. Which of the two is harder to cover with this network for an earthquake at 10–15 km depth? (We will see it in Part 2.)

## Part 2 — Focal mechanisms with SKHASH

**SKHASH** (Skoumal et al., 2024; based on HASH, Hardebeck & Shearer, 2002) searches all the focal mechanisms that are compatible with the **P first-motion polarities** you measured, given the source–station geometry (azimuth and take-off angle computed beforehand in a 1D velocity model). It returns a preferred mechanism, the uncertainty of the nodal planes and a quality grade (A best … D worst).

The published catalog was made with the same program, but with polarities read by a neural network, weights, and S/P amplitude ratios. Here **you** read the polarities, and at the end you compare your solution with the published one.

**The 2025 sequence.** The figure below (next cell) shows the focal mechanisms of the 2025 sequence in the catalog of Núñez-Jara et al. (2026): (A) map colored by the time after the mainshock (pink: mainshock; dark green: before the mainshock), with the mapped faults (CeB: Central Basin, KB: Kumburgaz Basin, CH: Central High); (B) zoom on the rupture area (green box in A); (C) magnitude versus time since the mainshock, colored by faulting type. The three events of this part are among them: the M 3.8 foreshock 36 minutes before the mainshock (event 2815, dark green next to the mainshock), a strike-slip and a normal-faulting aftershock. About 70% of the mechanisms are strike-slip, with one nodal plane parallel to the Main Marmara Fault; most of the others are normal faulting.

In [ ]:
# The focal mechanisms of the 2025 sequence (catalog of Núñez-Jara et al., 2026)
from IPython.display import Image
display(Image(filename=str(CLASS_DIR/"figures"/"2025_sequence_focal_mechanisms.png"), width=1000))

**Workflow**

1. Check which stations recorded each event, and map them.
2. **Place the stations on the focal sphere** (2.1): how the azimuth and the take-off angle of a ray give its position, with a scheme, examples (`sphere_position`) and an exercise (`sphere_exercise`, `guess_stations`).
3. **An amphibious network** (2.2): where would ocean-bottom stations above the fault plot on the focal sphere, and what would they add?
4. **Choose the stations** worth picking with `select_stations("<event_id>")` (2.3): drop those with a low signal-to-noise ratio.
5. **Pick the polarities** with `pick_polarities("<event_id>")` (2.4): the notebook shows the stations one by one and you click **▲ up**, **▼ down** or **? unclear**. Every answer is saved immediately in `skhash/<event_id>/pol.csv`.
6. **Map your picks** with `plot_polarity_map("<event_id>")` (2.5): up, down and unclear stations on the map and on the focal sphere.
7. **Run SKHASH** with `run_skhash("<event_id>")` (2.6–2.7) and look at the result with `show_result("<event_id>")` (2.8), next to the published solution and the mapped faults.
8. **Change the stations** (2.9): build subsets with `make_subset(...)` (only near stations, only far stations, one side of the epicenter, a few random stations, the ocean-bottom stations…), run them and compare with `compare_runs([...])`.

**Input files (made by `make_skhash_2025.py`)**

| File | Content |
|---|---|
| `picks.csv` | P and S arrival times picked automatically (PhaseNet) for the catalog: they mark the arrivals on the waveforms |
| `mseed_2025/EV<event_id>_raw.mseed` | raw waveforms (counts, 100 Hz, three components) of all the stations, from 10 s before to 110 s after the origin time. File 5 of the data publication; the id has 5 digits, e.g. `EV02815_raw.mseed` |
| `skhash/<event_id>/eq_catalog.csv` | `time, latitude, longitude, depth, horz_uncert_km, vert_uncert_km, mag, event_id` — hypocenter of the event (do not edit) |
| `skhash/<event_id>/pol.csv` | `event_id, station, network, location, channel, p_polarity, takeoff, takeoff_uncertainty, azimuth, azimuth_uncertainty` — one line per station with a P pick (after section 2.3: per selected station; `pol_all.csv` keeps the complete list). Column `p_polarity` holds **your picks** (the picker writes it for you) |
| `skhash/<event_id>/control_file.txt` | SKHASH parameters of the event: `$catfile`, `$fpfile`, `$npolmin`, `$max_agap`, `$max_pgap`, … (do not edit, unless told so) |

In [ ]:
# ---------- events and catalog picks ----------
def load_events():
    if not EVENTS_CSV.exists():
        raise FileNotFoundError(f"{EVENTS_CSV} not found: run make_skhash_2025.py first (it writes events.csv, picks.csv, "
                                f"stations.csv and the skhash/ folders).")
    ev = pd.read_csv(EVENTS_CSV, dtype={"event_id": str})          # event_id as text: it is also the folder name
    ev["event_id"] = ev.event_id.str.strip()
    ev["time"] = ev.time.apply(UTCDateTime)
    return ev.set_index("event_id", drop=False)

EVENTS = load_events()
EVENT_IDS = list(EVENTS.event_id)
PICKS = pd.read_csv(PICKS_CSV, dtype={"event_id": str, "station": str, "location": str})
PICKS["time"] = PICKS.time.apply(UTCDateTime)
EVENTS[["time", "latitude", "longitude", "depth", "mag"]]

In [ ]:
# Which stations recorded each event?  (P pick in the catalog + waveforms in mseed_2025/EV<event_id>_raw.mseed)
AVAIL = {}
for ev_id in EVENT_IDS:
    ev = EVENTS.loc[ev_id]
    f = event_file(ev_id)
    with_data = {tr.stats.station for tr in read(str(f), headonly=True) if tr.stats.channel.endswith("Z")} if f.exists() else set()
    with_p = set(PICKS[(PICKS.event_id == ev_id) & (PICKS.phase == "P")].station)
    df = stations[stations.station.isin(with_p)].copy()
    df["waveform"] = df.station.isin(with_data)
    AVAIL[ev_id] = df
    print(f"{ev_id} (M {ev.mag:.1f}, {ev.time.strftime('%Y-%m-%d %H:%M')}): {len(df)} stations with a P pick, "
          f"{df.waveform.sum()} of them with waveforms" + ("" if f.exists() else f"  -> {f} is missing!"))

    fig = plt.figure(figsize=(8, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    add_faults(ax, transform=ccrs.PlateCarree())
    ax.scatter(df.longitude, df.latitude, marker="^", s=50, c=np.where(df.waveform, "C0", "0.7"), transform=ccrs.PlateCarree())
    ax.scatter(ev.longitude, ev.latitude, marker="*", s=180, c="gold", ec="k", transform=ccrs.PlateCarree())
    for _, s in df.iterrows():
        ax.text(s.longitude, s.latitude, s.station, fontsize=7, transform=ccrs.PlateCarree())
    ax.set_title(f"{ev_id}: stations with a P pick (grey: no waveform)")
    plt.show()

**Tools for Part 2.** Run the next cell: it only defines the functions used below (station selection, the polarity picker, maps). You do not need to read it.

In [ ]:
# --- helper code for Part 2 (waveforms, station selection, picker, maps): just run this cell ---
from IPython.display import clear_output, display
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm

POL_COLS = ["event_id", "station", "network", "location", "channel", "p_polarity", "takeoff",
            "takeoff_uncertainty", "azimuth", "azimuth_uncertainty"]
POL_TEXT = {1.0: "up (+1)", -1.0: "down (-1)", 0.0: "unclear (0)"}
SYMBOL = {1.0: dict(marker="^", fc="k", ec="k", s=70, label="up (+1)"),
          -1.0: dict(marker="v", fc="w", ec="k", s=70, label="down (-1)"),
          0.0: dict(marker="x", fc="0.5", ec="0.5", s=35, label="unclear (0)")}

# ---------------- pol.csv: read / write ----------------
def read_pol(folder, name="pol.csv"):
    """pol.csv (or pol_all.csv) of skhash/<folder>/ as a table (text columns stay text, e.g. location '03')."""
    pol = pd.read_csv(SKHASH_DIR/str(folder)/name, skipinitialspace=True,
                      dtype={c: str for c in ("event_id", "station", "network", "location", "channel")})
    pol.columns = [c.strip() for c in pol.columns]
    pol["p_polarity"] = pd.to_numeric(pol.p_polarity, errors="coerce").fillna(0.0)
    return pol

def write_pol(folder, pol, name="pol.csv"):
    """Write the file with the same layout as make_skhash.py (values separated by ', ')."""
    lines = [", ".join(POL_COLS)] + [", ".join(str(v) for v in row) for row in pol[POL_COLS].itertuples(index=False)]
    (SKHASH_DIR/str(folder)/name).write_text("\n".join(lines) + "\n")

def event_of(folder):
    """event_id of a folder in skhash/ (read from its eq_catalog.csv)."""
    cat = pd.read_csv(SKHASH_DIR/str(folder)/"eq_catalog.csv", skipinitialspace=True, dtype={"event_id": str})
    cat.columns = [c.strip() for c in cat.columns]
    return cat.event_id.iloc[0].strip()

OBS_STATIONS = pd.DataFrame(columns=["station", "longitude", "latitude"])         # ocean-bottom stations, section 2.2

def _add_geometry(df, ev_id):
    """Station coordinates (land stations and OBS) and epicentral distance (km)."""
    ev = EVENTS.loc[ev_id]
    coords = stations[["station", "latitude", "longitude"]]
    if len(OBS_STATIONS):
        coords = pd.concat([coords, OBS_STATIONS[["station", "latitude", "longitude"]].astype({"latitude": float, "longitude": float})])
    df = df.merge(coords.drop_duplicates("station"), on="station", how="left")
    df["dist_km"] = [dist_az(ev.latitude, ev.longitude, la, lo)[0] if pd.notna(la) else np.nan
                     for la, lo in zip(df.latitude, df.longitude)]
    return df

def pol_geometry(folder, name="pol.csv"):
    """pol.csv + station coordinates + epicentral distance (km)."""
    return _add_geometry(read_pol(folder, name), event_of(folder))

# ---------------- focal sphere: lower hemisphere, equal-area projection ----------------
def sphere_base(ax, title=None):
    ax.add_patch(Circle((0, 0), 1, fc="none", ec="k", lw=1.2)); ax.plot([0, 0], [1, 1.04], "k"); ax.text(0, 1.06, "N", ha="center", va="bottom")
    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.2, 1.2); ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=9)

def sphere_xy(az, takeoff_down):
    # azimuth and takeoff angle measured from the downward vertical -> equal-area lower-hemisphere projection
    az, i = np.asarray(az, float), np.asarray(takeoff_down, float); up = i > 90
    trend = np.where(up, (az + 180) % 360, az); inc = np.where(up, 180 - i, i); r = np.sqrt(2)*np.sin(np.radians(inc)/2)
    return r*np.sin(np.radians(trend)), r*np.cos(np.radians(trend))

def nodal_xy(strike, dip, n=181):
    s, d = np.radians(strike), np.radians(dip); al = np.linspace(0, np.pi, n)
    es = np.array([np.cos(s), np.sin(s), 0.0]); ed = np.array([np.cos(d)*np.cos(s+np.pi/2), np.cos(d)*np.sin(s+np.pi/2), np.sin(d)])
    v = np.cos(al)[:, None]*es + np.sin(al)[:, None]*ed
    inc = np.degrees(np.arccos(np.clip(v[:, 2], -1, 1))); trend = np.degrees(np.arctan2(v[:, 1], v[:, 0])); r = np.sqrt(2)*np.sin(np.radians(inc)/2)
    return r*np.sin(np.radians(trend)), r*np.cos(np.radians(trend))

def _label(v, reviewed=None):
    # zeros are "unclear" for the picker (which knows the stations not seen yet), "unclear / not picked" elsewhere
    return "unclear / not picked (0)" if (v == 0 and reviewed is None) else SYMBOL[v]["label"]

def plot_picks_on_sphere(ax, pol, highlight=None, reviewed=None, title=None):
    """Stations of pol on the focal sphere with their polarity. reviewed: stations already seen by the picker."""
    sphere_base(ax, title)
    x, y = sphere_xy(pol.azimuth, 180 - pol.takeoff)                     # SKHASH takeoff: 0 = up, 180 = down
    sign = np.sign(pol.p_polarity.to_numpy())
    todo = np.zeros(len(pol), bool) if reviewed is None else (~pol.station.isin(reviewed)).to_numpy() & (sign == 0)
    if todo.any():
        ax.scatter(x[todo], y[todo], s=12, c="0.8", zorder=2, label="not picked yet")
    for v, sym in SYMBOL.items():
        m = (sign == v) & ~todo
        if m.any():
            ax.scatter(x[m], y[m], marker=sym["marker"], s=sym["s"]*0.7, facecolors=sym["fc"], edgecolors=sym["ec"],
                       linewidths=1.0, zorder=3, label=f"{_label(v, reviewed)}: {m.sum()}")
    if highlight is not None and (pol.station == highlight).any():
        k = int(np.flatnonzero((pol.station == highlight).to_numpy())[0])
        ax.scatter(x[k], y[k], s=260, facecolors="none", edgecolors="r", linewidths=2, zorder=4)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, fontsize=7, frameon=False)

# ---------------- waveforms, catalog picks, signal-to-noise ratio, automatic onset ----------------
_ST_CACHE = {}
def _event_stream(ev_id):
    """All the waveforms of one event: mseed_2025/EV<event_id>_raw.mseed (read once, then kept in memory)."""
    ev_id = str(ev_id)
    if ev_id not in _ST_CACHE:
        f = event_file(ev_id)
        if not f.exists():
            raise FileNotFoundError(f"waveform file {f} not found: run the download cell of 'Prepare the workspace' "
                                    f"(or delete {DATA_ROOT} and run it again)")
        st = read(str(f))
        known = set(stations.station)                       # stations.csv: GONAF (GF) is not in it
        _ST_CACHE[ev_id] = Stream([tr for tr in st if tr.stats.station in known])
    return _ST_CACHE[ev_id]

def _station_stream(ev_id, station):
    """The three components of one station (raw counts)."""
    st = _event_stream(ev_id).select(station=station).copy()
    st.merge(method=1, fill_value="interpolate")
    return st.sort(["location", "channel"])

def _z_traces(ev_id, station):
    """Vertical trace(s) of one station."""
    return _station_stream(ev_id, station).select(channel="*Z")

def pick_time(ev_id, station, phase="P"):
    """Arrival time of the catalog pick (PhaseNet) of a phase at a station, or None."""
    p = PICKS[(PICKS.event_id == str(ev_id)) & (PICKS.station == station) & (PICKS.phase == phase)]
    return p.time.iloc[0] if len(p) else None

def _prepare(tr, fmin, fmax, zerophase=False, corners=2):
    x = tr.copy(); x.detrend("demean"); x.detrend("linear"); x.taper(0.05, side="left")
    if fmin is not None:
        x.filter("bandpass", freqmin=fmin, freqmax=fmax, corners=corners, zerophase=zerophase)
    return x

SNR_WINDOW = 0.15      # s after the P pick: the first motion, not the largest (later) phase

def _snr(x, t, window=None):
    """S/N = largest amplitude in the first `window` s (default SNR_WINDOW) after the P pick / standard deviation of the
    noise in the 1.5 s before it."""
    window = SNR_WINDOW if window is None else window
    pre, post = x[(t > -1.5) & (t < -0.05)], x[(t >= 0) & (t <= window)]
    noise = pre.std() if len(pre) > 1 else np.nan
    return (np.abs(post).max()/max(noise, 1e-20) if len(post) else np.nan), noise

def _aic_onset(x, t, win=(-1.0, 1.0)):
    """Onset (s, relative to the P pick) found by an automatic AIC picker (Maeda, 1985) around the pick."""
    w = (t >= win[0]) & (t <= win[1]); y, tt = x[w], t[w]; n = len(y)
    if n < 10: return np.nan
    k = np.arange(2, n - 2); c1, c2 = np.cumsum(y), np.cumsum(y*y)
    v1 = c2[k - 1]/k - (c1[k - 1]/k)**2
    v2 = (c2[-1] - c2[k - 1])/(n - k) - ((c1[-1] - c1[k - 1])/(n - k))**2
    aic = k*np.log(np.maximum(v1, 1e-30)) + (n - k - 1)*np.log(np.maximum(v2, 1e-30))
    return float(tt[k[np.argmin(aic)]])

# ---------------- station selection ----------------
def station_quality(event_id, fmin=1.0, fmax=15.0, snr_window=None):
    """For every station of the event: its best vertical sensor, that sensor's S/N and the automatic (AIC) onset."""
    ev_id = str(event_id)
    base = read_pol(ev_id, "pol_all.csv" if (SKHASH_DIR/ev_id/"pol_all.csv").exists() else "pol.csv")
    rows = []
    for sta in base.station:
        best = dict(station=sta, best_sensor=None, n_sensors=0, snr=np.nan, aic_onset_s=np.nan)
        st, tp = _z_traces(ev_id, sta), pick_time(ev_id, sta, "P"); best["n_sensors"] = len(st)
        for tr in (st if tp is not None else []):
            flt = _prepare(tr, fmin, fmax); t = flt.times(reftime=tp)
            snr, _ = _snr(flt.data, t, snr_window)
            if np.isfinite(snr) and not (snr <= best["snr"]):
                best.update(best_sensor=f"{tr.stats.location or '--'}.{tr.stats.channel}", snr=snr, aic_onset_s=_aic_onset(flt.data, t))
        rows.append(best)
    return pd.DataFrame(rows)

def _ray(az, takeoff):
    a, i = np.radians(az), np.radians(takeoff)
    return np.array([np.sin(i)*np.cos(a), np.sin(i)*np.sin(a), -np.cos(i)])

def select_stations(event_id, snr_min=10, snr_window=None, thin={}, min_spacing_km=5, min_sep_deg=None, show=True):
    """Choose the stations worth picking for one event and write them to skhash/<event_id>/pol.csv.
      1. drop the stations whose best vertical sensor has S/N < snr_min (S/N measured in the first snr_window s after
         the P pick, default SNR_WINDOW = 0.15 s);
      2. optional, thin dense networks: e.g. thin={"KO": 10} keeps the 10 stations of network KO with the highest S/N
         that are at least min_spacing_km apart (default {}: no thinning);
      3. optional, min_sep_deg: from the highest S/N down, drop any station whose ray leaves the source less than
         min_sep_deg degrees from the ray of a station already kept (near-duplicates on the focal sphere, any network).
    The complete list of stations is kept in pol_all.csv (written the first time); polarities already picked are kept."""
    ev_id = str(event_id); folder = SKHASH_DIR/ev_id
    if not (folder/"pol_all.csv").exists():
        shutil.copy(folder/"pol.csv", folder/"pol_all.csv")                     # complete list, made only once
    full, picked = read_pol(ev_id, "pol_all.csv"), read_pol(ev_id).set_index("station").p_polarity
    full["p_polarity"] = [picked.get(s, p) for s, p in zip(full.station, full.p_polarity)]   # keep the picks made so far
    write_pol(ev_id, full, "pol_all.csv")
    q = _add_geometry(full.merge(station_quality(ev_id, snr_window=snr_window), on="station", how="left"), ev_id)
    q["reason"] = np.where(q.snr >= snr_min, "", f"S/N < {snr_min:g}")
    for net, n_keep in thin.items():                                             # thin dense networks
        kept = []
        for i in q[(q.network == net) & (q.reason == "")].sort_values("snr", ascending=False).index:
            if len(kept) < n_keep and all(dist_az(q.latitude[i], q.longitude[i], q.latitude[j], q.longitude[j])[0] >= min_spacing_km
                                          for j in kept):
                kept.append(i)
            else:
                q.loc[i, "reason"] = f"{net} thinned"
    if min_sep_deg:                                                              # near-duplicate rays, any network
        kept = []
        for i in q[q.reason == ""].sort_values("snr", ascending=False).index:
            v = _ray(q.azimuth[i], q.takeoff[i])
            if all(np.degrees(np.arccos(min(1.0, abs(v @ _ray(q.azimuth[j], q.takeoff[j]))))) >= min_sep_deg for j in kept):
                kept.append(i)
            else:
                q.loc[i, "reason"] = "near-duplicate ray"
    q["keep"] = q.reason == ""
    write_pol(ev_id, q[q.keep])
    k = q[q.keep]
    print(f"{ev_id}: {len(q)} stations with data -> {len(k)} kept for picking (written to {folder/'pol.csv'})")
    for reason, g in q[~q.keep].groupby("reason"):
        print(f"   dropped, {reason} ({len(g)}): " + ", ".join(g.station))
    for net in thin:
        kn = k[k.network == net]
        if len(kn): print(f"   {net} kept: " + ", ".join(f"{r.station} (S/N {r.snr:.0f})" for r in kn.itertuples()))
    ok = (k.aic_onset_s.abs() <= 0.1).sum()
    print(f"   automatic (AIC) onset within ±0.1 s of the catalog P pick for {ok} of the {len(k)} kept stations")
    if show: plot_selection(ev_id, q, f"S/N ≥ {snr_min:g}" + "".join(f", {n} best {net}" for net, n in thin.items())
                            + (f", rays ≥ {min_sep_deg:g}° apart" if min_sep_deg else ""))
    return q

# ---------------- maps ----------------
def _base_map(fig, spec, ev, lons, lats, rings=(25, 50, 100, 150), pad=0.25):
    """Map (conformal: true azimuths) with coastlines, distance rings and the epicenter."""
    pc = ccrs.PlateCarree()
    ax = fig.add_subplot(spec, projection=ccrs.Mercator())
    lons, lats = np.r_[lons, ev.longitude], np.r_[lats, ev.latitude]
    ax.set_extent([np.nanmin(lons) - pad, np.nanmax(lons) + pad, np.nanmin(lats) - pad, np.nanmax(lats) + pad], crs=pc)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, edgecolor="0.45")
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="0.6")
    add_faults(ax, transform=pc)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4); gl.top_labels = gl.right_labels = False
    for r_km in rings:
        la, lo = zip(*[offset_point(ev.latitude, ev.longitude, r_km, a) for a in np.linspace(0, 360, 181)])
        ax.plot(lo, la, "k--", lw=0.5, alpha=0.5, transform=pc)
        ax.text(lo[90], la[90], f"{r_km} km", fontsize=7, ha="center", va="top", color="0.3", transform=pc)
    star(ax, ev.latitude, ev.longitude, size=320, transform=pc)
    return ax, pc

def plot_selection(ev_id, q, crit=""):
    """Kept stations colored by S/N, dropped ones in grey, on the map and on the focal sphere."""
    ev = EVENTS.loc[ev_id]; k = q[q.keep]
    fig = plt.figure(figsize=(13, 6)); gs = fig.add_gridspec(1, 2, width_ratios=[1.8, 1], wspace=0.05)
    ax, pc = _base_map(fig, gs[0], ev, q.longitude, q.latitude)
    axs = fig.add_subplot(gs[1]); sphere_base(axs, "the same stations on the focal sphere\n(lower hemisphere, equal area)")
    xs, ys = sphere_xy(q.azimuth, 180 - q.takeoff)
    norm = LogNorm(vmin=max(1, k.snr.min() if len(k) else 1), vmax=max(q.snr.max(), 10))
    m = q.keep.to_numpy()
    sc = ax.scatter(k.longitude, k.latitude, c=k.snr, cmap="viridis", norm=norm, s=40, marker="o", edgecolors="k",
                    linewidths=0.5, transform=pc, zorder=4, label=f"kept ({m.sum()})")
    axs.scatter(xs[m], ys[m], c=q.snr[m], cmap="viridis", norm=norm, s=36, edgecolors="k", linewidths=0.5, zorder=3)
    drop_style = {"S/N": dict(marker="x", c="0.4", s=28, linewidths=1.0),
                  "thinned": dict(marker="o", facecolors="none", edgecolors="0.35", s=14, linewidths=0.7),
                  "near-duplicate": dict(marker="+", c="0.4", s=36, linewidths=1.0)}
    for key, sty in drop_style.items():                                          # dropped stations drawn on top
        d = (~q.keep & q.reason.str.contains(key, regex=False)).to_numpy()
        if d.any():
            ax.scatter(q.longitude[d], q.latitude[d], transform=pc, zorder=5, label=f"dropped: {q.reason[d].iloc[0]} ({d.sum()})", **sty)
            axs.scatter(xs[d], ys[d], zorder=4, **sty)
    ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
    cb = fig.colorbar(sc, ax=axs, orientation="horizontal", fraction=0.05, pad=0.02, label=f"S/N of the best vertical sensor (first {SNR_WINDOW:g} s after the P pick)")
    ticks = [t for t in (5, 10, 20, 50, 100, 200, 500, 1000) if norm.vmin <= t <= norm.vmax]
    cb.set_ticks(ticks); cb.set_ticklabels([str(t) for t in ticks]); cb.minorticks_off()
    ax.set_title(f"{ev_id}: {m.sum()} of {len(q)} stations kept ({crit})", fontsize=10)
    plt.show()

def plot_polarity_map(folder, labels=False, rings=(25, 50, 100, 150), pad=0.25):
    """Map and focal sphere of the polarities in skhash/<folder>/pol.csv, i.e. the stations used by that SKHASH run.
    For a subset folder, the stations of the event that are NOT in the subset are drawn as small grey circles."""
    folder = str(folder); ev_id = event_of(folder); ev = EVENTS.loc[ev_id]
    pol = pol_geometry(folder)
    allp = pol_geometry(ev_id) if folder != ev_id and (SKHASH_DIR/ev_id/"pol.csv").exists() else pol
    out = allp[~allp.station.isin(pol.station)]                                  # not used in this run
    fig = plt.figure(figsize=(13, 6)); gs = fig.add_gridspec(1, 2, width_ratios=[1.8, 1], wspace=0.05)
    ax, pc = _base_map(fig, gs[0], ev, allp.longitude, allp.latitude, rings, pad)
    if len(out):
        ax.scatter(out.longitude, out.latitude, s=22, marker="o", facecolors="none", edgecolors="0.6", linewidths=0.8,
                   transform=pc, zorder=4, label=f"not in this run: {len(out)}")
    sign = np.sign(pol.p_polarity.to_numpy())
    for v, sym in SYMBOL.items():
        m = sign == v
        if m.any():
            ax.scatter(pol.longitude[m], pol.latitude[m], marker=sym["marker"], s=sym["s"], facecolors=sym["fc"],
                       edgecolors=sym["ec"], linewidths=1.0, transform=pc, zorder=5, label=f"{_label(v)}: {m.sum()}")
    if labels:
        for r in pol.itertuples():
            ax.text(r.longitude + 0.02, r.latitude + 0.02, r.station, fontsize=6.5, transform=pc, zorder=6)
    ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
    picked = pol[sign != 0]
    info = (f"{len(picked)} stations with a polarity, {picked.dist_km.min():.0f}–{picked.dist_km.max():.0f} km, "
            f"azimuthal gap {azimuthal_gap(picked.azimuth):.0f}°") if len(picked) else "no polarities picked yet"
    ax.set_title(f"{folder}\n{info}", fontsize=10)
    plot_picks_on_sphere(fig.add_subplot(gs[1]), pol, title="the same picks on the focal sphere\n(lower hemisphere, equal area)")
    plt.show()

# ---------------- the station figure and the interactive picker ----------------
def plot_station(ev_id, station, pol=None, fmin=1.0, fmax=15.0, zoom=(-0.2, 0.3), gain=1, reviewed=None, title_prefix=""):
    """Three components around the P and S picks, zoom on the P onset (vertical) and the focal sphere with the picks
    made so far. gain > 1 magnifies the zoom panel vertically (large later swings are clipped)."""
    st, tp, ts = _station_stream(ev_id, station), pick_time(ev_id, station, "P"), pick_time(ev_id, station, "S")
    zs = st.select(channel="*Z")
    if not len(zs) or tp is None:
        print(f"{station}: " + ("no vertical trace in " + str(event_file(ev_id)) if not len(zs) else "no P pick")); return None
    pol = pol_geometry(ev_id) if pol is None else pol
    row = pol[pol.station == station].iloc[0] if (pol.station == station).any() else None
    n = len(zs)                                                                # usually 1 (several if a station has 2 sensors)
    fig = plt.figure(figsize=(12.5, 5.3 + 0.75*(n - 1)))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.3, 1], height_ratios=[1.2, 1.5], hspace=0.45/(1 + 0.15*(n - 1)), wspace=0.12)
    ax_full, ax_zoom, ax_sph = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[:, 1])
    # --- top: the three components of the first sensor, same scale, from 3 s before P to 4 s after S
    t_end = (ts - tp) + 4 if ts is not None else 15.0
    z0 = zs[0]
    comps = [z0] + [tr for tr in st.select(location=z0.stats.location)
                    if tr.stats.channel[:2] == z0.stats.channel[:2] and not tr.stats.channel.endswith("Z")][:2]
    xs = [_prepare(tr, fmin, fmax) for tr in comps]
    ts_ = [x.times(reftime=tp) for x in xs]
    norm = max(np.abs(x.data[(t >= -3) & (t <= t_end)]).max() for x, t in zip(xs, ts_)) or 1.0
    for k, (x, t) in enumerate(zip(xs, ts_)):
        w = (t >= -3) & (t <= t_end)
        ax_full.plot(t[w], x.data[w]/norm*0.95 - 2.0*k, color="k" if k == 0 else "0.45", lw=0.6)
        ax_full.text(-2.95, -2.0*k + 0.45, x.stats.channel, fontsize=7.5, color="k" if k == 0 else "0.4")
    ax_full.axvline(0, color="r", ls="--", lw=1); ax_full.text(0.05, 1.0, "P", color="r", fontsize=8, va="top")
    if ts is not None:
        ax_full.axvline(ts - tp, color="C0", ls="--", lw=1); ax_full.text(ts - tp + 0.05, 1.0, "S", color="C0", fontsize=8, va="top")
    ax_full.axvspan(*zoom, color="C3", alpha=0.08)
    ax_full.set_xlim(-3, t_end); ax_full.set_ylim(-2.0*(len(xs) - 1) - 1.1, 1.1); ax_full.set_yticks([])
    ax_full.set_title(f"three components, same scale (band-pass {fmin:g}–{fmax:g} Hz, causal); "
                      "dashed: catalog P and S picks", fontsize=8.5, loc="left")
    # --- bottom: zoom on the P onset of the vertical sensor(s)
    for k, tr in enumerate(zs):
        raw, flt = _prepare(tr, None, None), _prepare(tr, fmin, fmax)
        t = flt.times(reftime=tp)
        w = (t >= zoom[0]) & (t <= zoom[1])
        if w.sum() < 2:
            print(f"{tr.id}: no data around the P pick"); continue
        snr, noise = _snr(flt.data, t)
        off = -2.4*k
        label = f"{tr.stats.location or '--'}.{tr.stats.channel}"
        a_f, a_r = max(np.abs(flt.data[w]).max(), 1e-20)/gain, max(np.abs(raw.data[w]).max(), 1e-20)/gain
        ax_zoom.fill_between(zoom, off - min(3*noise/a_f, 1.15), off + min(3*noise/a_f, 1.15), color="C1", alpha=0.18, lw=0)
        ax_zoom.plot(t[w], np.clip(raw.data[w]/a_r, -1.15, 1.15) + off, color="0.65", lw=0.8)
        ax_zoom.plot(t[w], np.clip(flt.data[w]/a_f, -1.15, 1.15) + off, "k", lw=1.3)
        ax_zoom.axhline(off, color="0.5", lw=0.5)
        ax_zoom.text(zoom[0] + 0.005, off + 0.62, f"{label}  S/N ≈ {snr:.0f}", ha="left", fontsize=7.5,
                     color="k" if snr >= 5 else "r")
    ax_zoom.axvline(0, color="r", ls="--", lw=1); ax_zoom.set_yticks([])
    ax_zoom.set_xlim(*zoom); ax_zoom.set_xlabel("time relative to the catalog P pick (s)")
    ax_zoom.set_title("zoom on the P onset (vertical): filtered (black), raw (grey), pre-P noise ±3σ (orange)"
                      + (f"   vertical zoom ×{gain}" if gain != 1 else ""), fontsize=8.5, loc="left")
    ax_zoom.set_ylim(-2.4*(n - 1) - 1.3, 1.3)
    plot_picks_on_sphere(ax_sph, pol, highlight=station, reviewed=reviewed, title="your picks so far")
    head = f"{title_prefix}{station}"
    if row is not None:
        cur = float(np.sign(row.p_polarity))
        cur_txt = "not picked yet" if (reviewed is not None and station not in reviewed and cur == 0) else POL_TEXT[cur]
        head += (f"   {row.network}.{row.location}.{row.channel}   distance {row.dist_km:.0f} km   azimuth {row.azimuth:.0f}°"
                 f"   take-off {row.takeoff:.0f}°   current: {cur_txt}")
    fig.suptitle(head, fontsize=10.5, fontweight="bold", x=0.02, ha="left")
    return fig

def show_station(ev_id, station, **kw):
    """Look at one station without picking (same figure as the picker)."""
    if plot_station(str(ev_id), station, **kw) is not None:
        plt.show()

def compare_filters(ev_id, station, fmin=1.0, fmax=10.0, corners=4, window=(-0.2, 0.3)):
    """The same P onset: raw, causal band-pass and zero-phase band-pass (same band and order)."""
    tr, tp = _z_traces(str(ev_id), station)[0], pick_time(str(ev_id), station, "P")
    fig, ax = plt.subplots(figsize=(9, 3.2))
    for lab, zp, c, lw in (("raw", None, "0.6", 1.0), ("causal (one-way) filter", False, "k", 1.5),
                           ("zero-phase (two-way) filter", True, "C3", 1.5)):
        x = _prepare(tr, None if zp is None else fmin, fmax, zerophase=bool(zp), corners=corners)
        t = x.times(reftime=tp); w = (t >= window[0]) & (t <= window[1])
        ax.plot(t[w], x.data[w]/np.abs(x.data[w]).max(), color=c, lw=lw, label=lab)
    ax.axvline(0, color="r", ls="--", lw=1); ax.axhline(0, color="0.5", lw=0.5); ax.set_yticks([])
    ax.set_xlabel("time relative to the catalog P pick (s)"); ax.legend(fontsize=8, loc="upper left")
    ax.set_title(f"{station} ({tr.id}): band-pass {fmin:g}–{fmax:g} Hz, {corners} corners", fontsize=9)
    plt.tight_layout(); plt.show()

_KEYS = {"u": 1.0, "up": 1.0, "+": 1.0, "+1": 1.0, "1": 1.0, "d": -1.0, "down": -1.0, "-": -1.0, "-1": -1.0,
         "0": 0.0, "?": 0.0, "x": 0.0, "n": 0.0}
_GAINS = (1, 3, 10)
_WIDE = (-0.5, 1.0)                    # time window of the zoom panel after "w" (seconds around the P pick)
_PICK_BUTTONS = [("▲ up (u)", "u", "primary"), ("▼ down (d)", "d", "primary"), ("? unclear (0)", "0", "warning"),
                 ("keep / skip", "", ""), ("↕ vertical zoom", "z", "info"), ("↔ wider window", "w", "info"),
                 ("◀ back", "b", ""), ("■ stop", "q", "")]

def pick_polarities(folder, order="distance", max_dist_km=None, stations=None, redo=False, buttons=True, **plot_kw):
    """Show the stations of skhash/<folder>/pol.csv one by one and ask for the P polarity: click the buttons under
    the title (or, with buttons=False or without ipywidgets, type the answer). Every answer is written to pol.csv
    immediately. order: "distance" (default), "azimuth" or "file"."""
    folder = str(folder); ev_id = event_of(folder); pol = pol_geometry(folder)
    log_file = SKHASH_DIR/folder/"picking_log.csv"
    reviewed = set(pd.read_csv(log_file, dtype=str).station) if log_file.exists() else set()
    todo, missing = (pol if max_dist_km is None else pol[pol.dist_km <= max_dist_km]), []
    if stations is not None:
        wanted = [s.strip().upper() for s in stations]
        todo = todo[todo.station.isin(wanted)]
        missing = sorted(set(wanted) - set(todo.station))
    elif not redo:
        todo = todo[~todo.station.isin(reviewed)]
    if order in ("distance", "azimuth"):
        todo = todo.sort_values("dist_km" if order == "distance" else "azimuth")
    queue = list(todo.station)
    if not queue:
        print(f"{folder}: nothing to pick - every station has been reviewed already.\n"
              f"Use redo=True to go through all of them again, or stations=[...] to re-check some."); return
    windows = (plot_kw.pop("zoom", (-0.2, 0.3)), _WIDE)
    S = dict(i=0, g=0, wz=0, shown=None)                                       # position in the queue, zoom state

    def current():
        sta = queue[S["i"]]
        cur = float(np.sign(pol.loc[pol.station == sta, "p_polarity"].iloc[0]))
        keep = f"keep {POL_TEXT[cur]}" if (sta in reviewed or cur != 0) else "unclear (0)"
        return sta, cur, keep

    def plot():
        """Plot the current station, skipping stations without data; False when the queue is finished."""
        while S["i"] < len(queue):
            sta = queue[S["i"]]
            if sta != S["shown"]: S.update(g=0, wz=0, shown=sta)              # every station starts without zoom
            if plot_station(ev_id, sta, pol=pol, reviewed=reviewed, gain=_GAINS[S["g"]], zoom=windows[S["wz"]],
                            title_prefix=f"[{S['i'] + 1}/{len(queue)}]  ", **plot_kw) is not None:
                plt.show(); return True
            S["i"] += 1
        return False

    def answer(ans):
        """Apply one answer (u, d, 0, '' = keep, z, w, b, q). Returns False to stop."""
        sta, cur, keep = current()
        if ans in ("q", "quit"): return False
        if ans == "z": S["g"] = (S["g"] + 1) % len(_GAINS)                      # same station, other vertical zoom
        elif ans == "w": S["wz"] = (S["wz"] + 1) % len(windows)                 # same station, other time window
        elif ans in ("b", "back"): S["i"] = max(S["i"] - 1, 0)
        else:
            val = cur if ans == "" else _KEYS[ans]
            pol.loc[pol.station == sta, "p_polarity"] = val
            write_pol(folder, pol)                                            # saved after every answer
            reviewed.add(sta)
            with open(log_file, "a") as f:
                if f.tell() == 0: f.write("station,p_polarity,time\n")
                f.write(f"{sta},{val},{UTCDateTime()}\n")
            S["i"] += 1
        return True

    def summary():
        s = np.sign(pol.p_polarity)
        print(f"{folder}: {int((s > 0).sum())} up, {int((s < 0).sum())} down, {int((s == 0).sum())} unclear/not picked "
              f"({len(reviewed & set(pol.station))} of {len(pol)} stations reviewed). Saved in {SKHASH_DIR/folder/'pol.csv'}")
        if missing: print("not in this event's pol.csv:", ", ".join(missing))
        if S["i"] < len(queue): print("Stopped before the end: run the same cell again to continue where you stopped.")
        print(f"Next: plot_polarity_map('{folder}')")

    if buttons and widgets is not None:                                       # ---- buttons (JupyterLab, Jupyter Notebook, VS Code)
        out, info, btns = widgets.Output(), widgets.HTML(), []

        def show():
            out.clear_output(wait=True)
            more = None
            with out:
                more = plot()
                if not more:
                    summary()
            if more is None:                                                  # an error, shown in the output area
                info.value = "<b>Something went wrong</b> (message below). <i>keep / skip</i> goes to the next station."
            elif more:
                sta, cur, keep = current()
                info.value = (f"<b>{sta}</b> &nbsp;({S['i'] + 1} of {len(queue)}): is the first motion up, down or unclear? "
                              f"&nbsp; current value: {POL_TEXT[cur]} &nbsp;(<i>keep / skip</i> = {keep})")
            else:
                info.value = "<b>Done:</b> you reached the end of the list."
                for b in btns: b.disabled = True

        def click(ans):
            with out:                                                         # errors appear in the output area
                if not answer(ans):
                    for b in btns: b.disabled = True
                    out.clear_output(wait=True); summary()
                    info.value = "<b>Stopped.</b> Run the cell again to continue where you stopped."
                    return
            show()

        for label, ans, style in _PICK_BUTTONS:
            b = widgets.Button(description=label, button_style=style, layout=widgets.Layout(width="auto"))
            b.on_click(lambda _b, a=ans: click(a))
            btns.append(b)
        display(widgets.VBox([info, widgets.HBox(btns, layout=widgets.Layout(flex_flow="row wrap")), out]))
        show()
        return

    while plot():                                                             # ---- typed answers
        sta, cur, keep = current()
        while True:
            ans = input(f"{sta}:  u = up | d = down | 0 = unclear | Enter = {keep} | "
                        f"z = vertical zoom | w = wider window | b = back | q = quit  > ").strip().lower()
            if ans in _KEYS or ans in ("", "z", "w", "b", "back", "q", "quit"): break
            print(f"   '{ans}' not understood")
        if not answer(ans): break
        clear_output(wait=True)
    summary()

### 2.1 From a station to a point on the focal sphere

Before picking, it helps to know **where each station plots on the focal sphere**: that is the part of the mechanism its polarity will constrain.

Imagine a small sphere around the hypocenter, the **focal sphere**. The P wave that reaches a station leaves the source along a ray, and the ray pierces the focal sphere at one point. Two angles give that point:

| Angle | Meaning | Where it comes from |
|---|---|---|
| **azimuth** φ | direction of the ray in map view, clockwise from north (0° = N, 90° = E, 180° = S, 270° = W) | epicenter → station, from the coordinates |
| **take-off angle** *i* | angle between the ray and the **upward** vertical when it leaves the source: 0° = straight up, 90° = horizontal, 180° = straight down (SKHASH convention, column `takeoff` of `pol.csv`) | ray tracing in the velocity model (`make_skhash_2025.py`) |
| plunge | the same as the take-off angle, but measured from the horizontal: plunge = *i* − 90° (positive = downwards) | — |

> **Conventions differ between programs.** SKHASH uses the convention of HASH: the take-off angle is measured from the **upward** vertical (HASH manual: "degrees from vertical: up=0; <90 up-going; >90 down-going"), and so does `pol.csv`. Other programs measure it from the **downward** vertical (0° = down, 180° = up, e.g. FOCI). Converting is simple: *i*<sub>from down</sub> = 180° − *i*<sub>from up</sub>. The take-off angle is a polar angle (from 0° to 180°); the direction around the vertical is given by the azimuth, so "clockwise" or "anticlockwise" only matters when it is drawn in a 2-D section. Mixing up the two conventions mirrors every ray through the horizontal plane (up-going rays become down-going): for a pure strike-slip or a pure dip-slip mechanism the predicted polarities hardly change, so the error can go unnoticed, but oblique mechanisms come out wrong. Section 2.7 has a test that checks it with SKHASH.

**The projection.** We look at the focal sphere from above, keep its **lower half** only and flatten it with the equal-area (Schmidt) projection:

- the point lies in the direction of the azimuth φ (north up, east right);
- its distance from the center is $r = \sqrt{2}\,\sin(\theta/2)$, where $\theta = 180° - i$ is the angle from the **downward** vertical. A ray that goes straight down plots at the center ($r = 0$), a horizontal ray on the rim ($r = 1$).

**Upgoing rays** (*i* < 90°, the close stations) pierce the *upper* half of the sphere. They are drawn at the **other end of the same line through the center**: azimuth φ + 180°, take-off 180° − *i*. This is allowed because the P radiation pattern is symmetric through the center: the polarity is the same at both ends of the line. So a close station to the **north-east** plots in the **south-west** of the focal sphere!

The scheme below follows four rays that leave the source in the same vertical plane: their take-off angles (a), their azimuths in map view (b) and where they plot on the focal sphere (c).

In [ ]:
# --- helper code: from a ray (azimuth, take-off angle) to a point on the focal sphere. Run it: it draws the scheme ---
from matplotlib.patches import Arc, Wedge

_COMPASS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
def _compass(az):
    return _COMPASS[int((az % 360)/22.5 + 0.5) % 16]

def sphere_point(azimuth, takeoff):
    """Where a ray plots on the lower-hemisphere, equal-area projection (radius 1 = horizontal ray).
    azimuth: degrees clockwise from north; takeoff: 0 = up, 90 = horizontal, 180 = down (SKHASH convention).
    Returns x (east), y (north), the azimuth and take-off that are plotted, and True if the ray was flipped."""
    az, tk = float(azimuth) % 360, float(takeoff)
    flipped = tk < 90                                       # upgoing ray: use the other end of the same line
    if flipped:
        az, tk = (az + 180) % 360, 180 - tk
    r = np.sqrt(2)*np.sin(np.radians(180 - tk)/2)            # 180 - tk = angle from the downward vertical
    return r*np.sin(np.radians(az)), r*np.cos(np.radians(az)), az, tk, flipped

def explain_point(azimuth, takeoff):
    """The steps of the projection, in words."""
    x, y, az, tk, flipped = sphere_point(azimuth, takeoff)
    r = np.hypot(x, y)
    where = "at the center" if r < 0.05 else "on the rim" if r > 0.97 else f"{r*100:.0f}% of the way from the center to the rim"
    txt = f"azimuth {azimuth % 360:.0f}° ({_compass(azimuth)}), take-off {takeoff:.0f}° (plunge {takeoff - 90:+.0f}°):\n"
    if flipped:
        txt += (f"   the ray goes UP (take-off < 90°) -> use the other end of the line: azimuth {az:.0f}° ({_compass(az)}), "
                f"take-off {tk:.0f}°\n")
    else:
        txt += "   the ray goes down (or horizontally) -> plot it as it is\n"
    txt += f"   angle from the downward vertical = 180° - {tk:.0f}° = {180 - tk:.0f}° -> r = √2·sin({180 - tk:.0f}°/2) = {r:.2f}\n"
    txt += (f"   -> the point is {where}, towards {_compass(az)}" if r >= 0.05
            else "   -> the point is at the center (the azimuth does not matter)")
    return txt

def side_view(ax, rays, title=None, arc=None):
    """Vertical section through the source: right = towards the azimuth phi, left = towards phi + 180°.
    rays: list of (label, takeoff, side, color), side = +1 (towards phi) or -1 (towards phi + 180°)."""
    ax.add_patch(Wedge((0, 0), 1, 180, 360, fc="0.92", ec="none"))              # lower half of the sphere
    ax.add_patch(Circle((0, 0), 1, fc="none", ec="k", lw=1.2))
    ax.plot([-1.3, 1.3], [0, 0], color="0.6", lw=0.6, ls=":"); ax.plot([0, 0], [-1.3, 1.3], color="0.6", lw=0.6, ls=":")
    ax.text(0, 1.33, "up", ha="center", va="bottom", fontsize=8, color="0.4")
    ax.text(0.05, -0.55, "lower\nhemisphere", ha="left", fontsize=7.5, color="0.45")
    for label, tk, side, c in rays:
        dx, dy = side*np.sin(np.radians(tk)), np.cos(np.radians(tk))             # take-off from the upward vertical
        ax.annotate("", xy=(1.45*dx, 1.45*dy), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.6), zorder=4)
        ax.scatter(dx, dy, s=55, color=c, ec="k", zorder=5)
        ax.text(1.62*dx, 1.62*dy, label, color=c, fontsize=8, ha="center", va="center", fontweight="bold")
        if tk < 90:                                                             # upgoing: the other end of the line
            ax.plot([0, -dx], [0, -dy], ls="--", color=c, lw=1.1, zorder=3)
            ax.scatter(-dx, -dy, s=55, facecolors="w", edgecolors=c, linewidths=1.8, zorder=5)
    if arc is not None:                                                         # take-off angle of one ray
        tk, side, c = arc
        t1, t2 = (90 - tk, 90) if side > 0 else (90, 90 + tk)
        ax.add_patch(Arc((0, 0), 0.62, 0.62, theta1=t1, theta2=t2, color=c, lw=1.2))
        m = np.radians(90 - side*tk/2)
        ax.text(0.5*np.cos(m), 0.5*np.sin(m), f"i = {tk:.0f}°", color=c, fontsize=8, ha="center", va="center",
                bbox=dict(fc="w", ec="none", alpha=0.8, pad=0.5))
    ax.plot(0, 0, marker="*", ms=15, color="gold", mec="k", zorder=6)
    ax.set_xlim(-2.0, 2.0); ax.set_ylim(-1.75, 1.6); ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=9)

def sphere_scheme(phi=60):
    """Four rays in the vertical plane of azimuth phi: take-off angles, map view and focal sphere."""
    rays = [("A (20 km)", 67, +1, "C0"), ("B (60 km)", 95, +1, "C2"), ("C (135 km)", 130, +1, "C3"), ("D (25 km)", 70, -1, "C1")]
    dist = {"A": 20, "B": 60, "C": 135, "D": 25}
    fig = plt.figure(figsize=(14.5, 5.0)); gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1, 1], wspace=0.12)
    # (a) take-off angles
    ax = fig.add_subplot(gs[0])
    side_view(ax, rays, arc=(67, +1, "C0"),
              title=f"(a) vertical section through the source (not to scale)\nright: towards φ = {phi}° ({_compass(phi)}), "
                    f"left: towards φ + 180° = {phi + 180}° ({_compass(phi + 180)})")
    ax.text(-1.95, -1.72, "A, D: close stations, direct ray going up (i < 90°)\nB: refracted in the crust, leaves almost horizontally\n"
                          "C: refracted at the Moho, leaves downwards", fontsize=7.5, va="bottom", color="0.25")
    # (b) map view: azimuths
    axm = fig.add_subplot(gs[1])
    for label, tk, side, c in rays:
        az = phi if side > 0 else phi + 180; d = dist[label[0]]
        x, y = d*np.sin(np.radians(az)), d*np.cos(np.radians(az))
        axm.plot([0, x], [0, y], color=c, lw=1.0, ls="--")
        axm.scatter(x, y, marker="^", s=90, color=c, ec="k", zorder=4)
        axm.text(x + 6, y - 2, label[0], color=c, fontsize=9, fontweight="bold", va="top")
    axm.annotate("", xy=(0, 60), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="k", lw=1.2))
    axm.text(0, 63, "N", ha="center", va="bottom", fontsize=9)
    axm.add_patch(Arc((0, 0), 70, 70, theta1=90 - phi, theta2=90, color="k", lw=1.0))
    axm.text(14*np.sin(np.radians(phi/2)) + 2, 40*np.cos(np.radians(phi/2)), f"φ = {phi}°", fontsize=8.5)
    axm.plot(0, 0, marker="*", ms=15, color="gold", mec="k", zorder=6)
    axm.set_xlim(-60, 140); axm.set_ylim(-60, 100); axm.set_aspect("equal"); axm.grid(alpha=0.3, lw=0.4)
    axm.set_xlabel("east (km)"); axm.set_ylabel("north (km)")
    axm.set_title("(b) map view: the azimuth φ of each station\n(clockwise from north, epicenter → station)", fontsize=9)
    # (c) focal sphere
    axs = fig.add_subplot(gs[2]); sphere_base(axs, "(c) where they plot: lower hemisphere, equal area\n(filled: as they are; open: upgoing ray, flipped)")
    _takeoff_circles(axs)
    axs.plot([0, 1.05*np.sin(np.radians(phi))], [0, 1.05*np.cos(np.radians(phi))], color="0.6", lw=0.8, ls="--")
    axs.text(1.2*np.sin(np.radians(phi)), 1.2*np.cos(np.radians(phi)), f"φ = {phi}°", fontsize=7.5, color="0.4", ha="center")
    for label, tk, side, c in rays:
        x, y, az, tk2, fl = sphere_point(phi if side > 0 else phi + 180, tk)
        axs.scatter(x, y, s=70, facecolors="w" if fl else c, edgecolors=c if fl else "k", linewidths=1.8 if fl else 0.8, zorder=5)
        px, py = -np.cos(np.radians(az)), np.sin(np.radians(az))                 # label beside the point, across the radius
        axs.text(x + 0.11*px, y + 0.11*py, label[0], color=c, fontsize=9, fontweight="bold", ha="center", va="center")
    axs.text(0, -1.38, "center = straight down, rim = horizontal", ha="center", fontsize=7.5, color="0.35")
    plt.show()

def _takeoff_circles(ax):
    """Dotted circles of equal take-off angle on the focal sphere (lower hemisphere)."""
    for tk in (120, 150):
        r = np.sqrt(2)*np.sin(np.radians(180 - tk)/2)
        ax.add_patch(Circle((0, 0), r, fc="none", ec="0.7", lw=0.6, ls=":"))
        ax.text(0, -r + 0.02, f"i = {tk}°", fontsize=6.5, color="0.5", ha="center", va="bottom")
    ax.text(0, -0.98, "i = 90°", fontsize=6.5, color="0.5", ha="center", va="bottom")
    ax.text(0.03, 0.03, "i = 180°", fontsize=6.5, color="0.5", ha="left", va="bottom")

def sphere_position(azimuth, takeoff=None, plunge=None):
    """Where does a ray plot on the focal sphere? Give the azimuth and the take-off angle (0 = up, 180 = down),
    or the azimuth and the plunge (degrees below the horizontal, negative = upwards)."""
    takeoff = 90 + plunge if takeoff is None else takeoff
    print(explain_point(azimuth, takeoff))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.4), gridspec_kw=dict(width_ratios=[1.15, 1]))
    side_view(a1, [("ray", takeoff, +1, "C3")], arc=(takeoff, +1, "C3"),
              title=f"vertical section along the azimuth {azimuth % 360:.0f}°\n(right = towards {_compass(azimuth)}, the station side)")
    _sphere_points(a2, [(azimuth, takeoff)])
    plt.tight_layout(); plt.show()

def _sphere_points(ax, points, labels=None):
    """Points (azimuth, take-off) on the focal sphere; the last one is highlighted, with the station direction."""
    sphere_base(ax, "lower hemisphere, equal area\n(dashed: direction of the station; open = flipped)")
    _takeoff_circles(ax)
    for k, (az, tk) in enumerate(points):
        x, y, _, _, fl = sphere_point(az, tk); last = k == len(points) - 1; c = "C3" if last else "0.45"
        ax.scatter(x, y, s=80 if last else 40, facecolors="w" if fl else c, edgecolors=c if fl else "k",
                   linewidths=1.8 if fl else 0.8, zorder=5)
        ax.text(x + 0.06, y + 0.06, labels[k] if labels else str(k + 1), color=c, fontsize=8, fontweight="bold" if last else None)
    az = points[-1][0]
    ax.plot([0, 1.05*np.sin(np.radians(az))], [0, 1.05*np.cos(np.radians(az))], color="C3", lw=0.8, ls="--")


def _parse_ray(ans):
    """'45 120' (azimuth, take-off) or '45 p30' (azimuth, plunge) -> (azimuth, take-off), or None if not understood."""
    try:
        a, b = ans.strip().lower().replace(",", " ").split()
        az, tk = float(a), (90 + float(b[1:]) if b.startswith("p") else float(b))
    except ValueError:
        return None
    return (az, tk) if 0 <= tk <= 180 else None

_NOT_UNDERSTOOD = "   not understood: two numbers, e.g. 45 120 (take-off between 0 and 180) or 45 p30"

def _draw_exercise(pts):
    """The last point in a vertical section and all the points on the focal sphere."""
    az, tk = pts[-1]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.4), gridspec_kw=dict(width_ratios=[1.15, 1]))
    side_view(a1, [(f"{len(pts)}", tk, +1, "C3")], arc=(tk, +1, "C3"),
              title=f"point {len(pts)}: vertical section along the azimuth {az % 360:.0f}°\n(right = towards {_compass(az)})")
    _sphere_points(a2, pts)
    plt.tight_layout(); plt.show()
    print(f"point {len(pts)}: " + explain_point(az, tk))

def sphere_exercise(buttons=True):
    """Give an azimuth and a take-off angle (e.g. 45 120) or an azimuth and a plunge (45 p30): the point is added
    to the focal sphere, numbered in the order you gave them. With buttons: type in the box and click "Plot".
    With buttons=False (or without ipywidgets): type the answer when asked, q to stop."""
    pts = []
    if buttons and widgets is not None:
        box = widgets.Text(placeholder="e.g. 45 120   or   45 p30", description="azimuth, take-off:",
                           style={"description_width": "initial"}, layout=widgets.Layout(width="330px"))
        b_plot, b_clear, out = widgets.Button(description="Plot", button_style="primary"), widgets.Button(description="Clear the points"), widgets.Output()
        def on_plot(_):
            with out:
                ray = _parse_ray(box.value)
                if ray is None:
                    print(_NOT_UNDERSTOOD); return
                pts.append(ray); out.clear_output(wait=True); _draw_exercise(pts)
        def on_clear(_):
            pts.clear(); out.clear_output()
        b_plot.on_click(on_plot); b_clear.on_click(on_clear)
        display(widgets.VBox([widgets.HBox([box, b_plot, b_clear]), out]))
        return
    while True:
        ans = input("azimuth take-off (e.g. 45 120)  |  azimuth pPLUNGE (e.g. 45 p30)  |  q = stop  > ").strip().lower()
        if ans in ("q", "quit", ""):
            break
        ray = _parse_ray(ans)
        if ray is None:
            print(_NOT_UNDERSTOOD); continue
        pts.append(ray); clear_output(wait=True); _draw_exercise(pts)
    print(f"{len(pts)} points.")

def guess_stations(ev_id, names, buttons=True):
    """Distance, azimuth and take-off of some stations of an event; click the button (or press Enter) to see where
    they plot on the map and on the focal sphere."""
    ev_id = str(ev_id); name = "pol_all.csv" if (SKHASH_DIR/ev_id/"pol_all.csv").exists() else "pol.csv"
    pol = pol_geometry(ev_id, name); sel = pol.set_index("station").reindex([n.upper() for n in names]).dropna(subset=["azimuth"])
    missing = sorted(set(n.upper() for n in names) - set(sel.index))
    if missing: print("not recorded by this event:", ", ".join(missing))
    if sel.empty: return
    print(sel[["dist_km", "azimuth", "takeoff"]].round(0).astype(int).rename(columns=dict(
        dist_km="distance (km)", azimuth="azimuth (°)", takeoff="take-off (°)")).to_string())

    def reveal():
        ev = EVENTS.loc[ev_id]
        fig = plt.figure(figsize=(12, 5.2)); gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1], wspace=0.05)
        ax, pc = _base_map(fig, gs[0], ev, sel.longitude, sel.latitude, rings=(25, 50, 100, 150))
        for k, (sta, r) in enumerate(sel.iterrows()):
            c = f"C{k}"
            ax.plot([ev.longitude, r.longitude], [ev.latitude, r.latitude], color=c, lw=1, ls="--", transform=pc)
            ax.scatter(r.longitude, r.latitude, marker="^", s=90, color=c, ec="k", transform=pc, zorder=5)
            ax.text(r.longitude + 0.03, r.latitude + 0.03, sta, color=c, fontsize=9, fontweight="bold", transform=pc)
        ax.set_title(f"{ev_id}: the stations on the map", fontsize=10)
        axs = fig.add_subplot(gs[1]); sphere_base(axs, "... and on the focal sphere\n(open symbol: upgoing ray, flipped)")
        for k, (sta, r) in enumerate(sel.iterrows()):
            x, y, az, tk, fl = sphere_point(r.azimuth, r.takeoff); c = f"C{k}"
            axs.scatter(x, y, s=80, facecolors="w" if fl else c, edgecolors=c if fl else "k", linewidths=1.8 if fl else 0.8, zorder=5)
            axs.text(x + 0.06, y + 0.06, sta, color=c, fontsize=9, fontweight="bold")
        plt.show()
        for sta, r in sel.iterrows():
            print(f"{sta}: " + explain_point(r.azimuth, r.takeoff))

    if buttons and widgets is not None:
        print("Where do these stations plot on the focal sphere? Sketch it, then click the button to check.")
        b, out = widgets.Button(description="Show where they plot", button_style="primary"), widgets.Output()
        def on_click(_):
            b.disabled = True
            with out: reveal()
        b.on_click(on_click)
        display(widgets.VBox([b, out]))
        return
    input("Where do these stations plot on the focal sphere? Sketch it, then press Enter to check  > ")
    reveal()

sphere_scheme()

#### Exercise 2 — Predict first, then check

For each ray, decide where it plots (direction and distance from the center) *before* running the cell below:

1. azimuth 90°, take-off 90°: a horizontal ray towards the east;
2. azimuth 0°, take-off 180°: straight down;
3. azimuth 30°, take-off 130°: downwards, towards the north-north-east (like a distant station whose first P is refracted at the Moho);
4. azimuth 30°, take-off 60°: **upwards**, towards the north-north-east (like a close station).

In [ ]:
sphere_position(90, 90)      # 1. horizontal, towards the east
sphere_position(0, 180)      # 2. straight down
sphere_position(30, 130)     # 3. downwards, towards NNE
sphere_position(30, 60)      # 4. upwards, towards NNE      (also possible: sphere_position(30, plunge=-30))

#### Exercise 3 — Your turn

In the box of `sphere_exercise()`, type an azimuth and a take-off angle (e.g. `45 120`), or an azimuth and a plunge (`45 p30`), and click **Plot**: the point is added to the focal sphere. Predict the position before you click. (Without `ipywidgets`, type when asked and `q` to stop.)

- Which take-off angles plot on the rim? Which one plots at the center?
- Keep the take-off at 60° and change the azimuth: 0°, 90°, 180°, 270°. Where do the points go, and why?
- Find two different rays (azimuth, take-off) that plot at the **same** point.
- A station 20 km from the epicenter of an earthquake 10 km deep: which take-off angle, roughly, if the ray were a straight line? Where does it plot? And a station at 130 km whose first P is refracted at the Moho (take-off ≈ 130°)?

Then **real stations**: `guess_stations` lists the distance, azimuth and take-off angle of a few stations of the first event (from `pol.csv`). Sketch where they plot, click **Show where they plot** and compare. Why do ELBA and BAND, on opposite sides of the epicenter, swap sides on the focal sphere?

In [ ]:
sphere_exercise()          # no buttons (only a line of text like VBox(...))? use sphere_exercise(buttons=False)

In [ ]:
guess_stations(EVENT_IDS[0], ["ELBA", "BAND", "AVCA", "LAP"])     # try other stations of pol.csv too

### 2.2 An amphibious network: what would ocean-bottom stations add?

All the real stations are on land. The events are 10–14 km deep under the Sea of Marmara and the closest station is about 30 km away, so the rays leave the source almost horizontally (or downwards) and the stations fill only the **outer part** of the focal sphere (2.1). The center of the sphere — rays that leave the source steeply — stays empty, and that is where the **dip** of the nodal planes is decided.

Imagine an **amphibious network**: the land stations plus 12 ocean-bottom seismometers (OBS) on the sea floor, in two lines along the fault, about 5 km north and south of its trace (`OBS_STATIONS` in the cell below). For each event, the figure shows the network on the map and where every station plots on the focal sphere of the **published mechanism**: land stations in gray, OBS in blue with the first motion they would record (▲ up, ▽ down, × too close to a nodal plane to be read). The take-off angles of the OBS are computed with the same velocity model as for the land stations.

- Which part of the focal sphere do the OBS fill? Why?
- For which event do the OBS add the most: the strike-slip events (2815, 4009) or the normal-faulting event 8317? Look at where the OBS fall relative to the nodal planes.
- Which OBS are of no use for an event (×), and why? What does that tell you about where to deploy instruments?

In 2.9 you add the OBS to your own picks (with the polarities predicted by the published mechanism, as if they had been picked without error) and see how much the SKHASH solution improves.

In [ ]:
# --- helper code: an amphibious network (land stations + ocean-bottom seismometers). Just run this cell ---
from matplotlib.colors import ListedColormap
from obspy.imaging.beachball import aux_plane
if str(CLASS_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(CLASS_DIR.resolve()))
import make_skhash_2025 as PREP          # the 1D velocity model and ray tracer used for the take-off angles of pol.csv

# 12 ocean-bottom seismometers (OBS) on the sea floor (~1 km deep), in two lines along the Main Marmara Fault,
# about 5 km north (40.90°N) and south (40.79°N) of its trace, every 0.1° (~8.5 km) between 28.05°E and 28.55°E
OBS_DEPTH_KM = 1.0
OBS_STATIONS = pd.DataFrame([dict(station=f"OBS{k + 1:02d}", longitude=round(lon, 2), latitude=lat)
                             for k, (lat, lon) in enumerate((la, lo) for la in (40.90, 40.79) for lo in np.arange(28.05, 28.56, 0.1))])

def published_mechanism(ev_id):
    """(strike, dip, rake) of an event in the published catalog (mechanisms_catalog.csv), or None."""
    cat = pd.read_csv(CATALOG_CSV, dtype={"event_id": str})
    c = cat[cat.event_id.str.strip() == str(ev_id)]
    return None if c.empty else tuple(float(v) for v in c.iloc[0][["strike", "dip", "rake"]])

def radiation(sdr, azimuth, takeoff):
    """P radiation pattern of a double couple for rays (azimuth, take-off: 0 = up): + = compression (first motion up)."""
    n, u = fault_vectors(*sdr)
    i, a = np.radians(180 - np.asarray(takeoff, float)), np.radians(np.asarray(azimuth, float))
    g = np.stack([np.sin(i)*np.cos(a), np.sin(i)*np.sin(a), np.cos(i)], -1)          # ray (north, east, down)
    return 2*(g @ n)*(g @ u)

def obs_ray(ev_id, lon, lat, receiver_depth_km=OBS_DEPTH_KM):
    """Epicentral distance (km), azimuth and take-off angle (same 1D model as the land stations) of a point."""
    ev = EVENTS.loc[str(ev_id)]
    d, az = dist_az(ev.latitude, ev.longitude, lat, lon)
    return d, az % 360, PREP.first_P(d, ev.depth, receiver_depth_km)["takeoff"]

def obs_pol_rows(ev_id, obs=None):
    """pol.csv lines of the OBS for one event, with the polarity predicted by the published mechanism
    (0 = too close to a nodal plane to be read)."""
    obs = OBS_STATIONS if obs is None else obs
    sdr, rows = published_mechanism(ev_id), []
    for v in obs.itertuples():
        d, az, tk = obs_ray(ev_id, v.longitude, v.latitude)
        rad = radiation(sdr, az, tk) if sdr else 0.0
        rows.append(dict(event_id=str(ev_id), station=v.station, network="OB", location="--", channel="HHZ",
                         p_polarity=0.0 if abs(rad) < 0.1 else float(np.sign(rad)), takeoff=round(tk, 1), takeoff_uncertainty=5.0,
                         azimuth=round(az, 1), azimuth_uncertainty=5.0, latitude=v.latitude, longitude=v.longitude,
                         dist_km=d, amplitude=rad))
    return pd.DataFrame(rows)

def _beachball_background(ax, sdr, n=301):
    """Compressional quadrants (shaded) and nodal planes of a mechanism on the focal sphere (lower hemisphere)."""
    x, y = np.meshgrid(np.linspace(-1, 1, n), np.linspace(-1, 1, n)); r = np.hypot(x, y)
    theta = 2*np.degrees(np.arcsin(np.clip(r/np.sqrt(2), 0, 1)))                 # angle from the downward vertical
    rad = radiation(sdr, np.degrees(np.arctan2(x, y)), 180 - theta)
    ax.imshow(np.where(r <= 1, (rad > 0).astype(float), np.nan), extent=(-1, 1, -1, 1), origin="lower",
              cmap=ListedColormap(["white", "#f6cfc9"]), vmin=0, vmax=1, zorder=0)
    s2, d2, _ = aux_plane(*sdr)
    for ss, dd in ((sdr[0], sdr[1]), (s2, d2)):
        ax.plot(*nodal_xy(ss, dd), color="0.35", lw=1.2, zorder=1)

def show_amphibious(ev_id, half_width_km=45):
    """Map of the amphibious network and the focal sphere of the published mechanism with land stations (grey)
    and OBS (blue: the first motion they would record)."""
    ev_id = str(ev_id); ev = EVENTS.loc[ev_id]; sdr = published_mechanism(ev_id); pc = ccrs.PlateCarree()
    land = pol_geometry(ev_id, "pol_all.csv" if (SKHASH_DIR/ev_id/"pol_all.csv").exists() else "pol.csv")
    obs = obs_pol_rows(ev_id)
    fig = plt.figure(figsize=(14, 6.2)); gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1], wspace=0.06)
    ax = fig.add_subplot(gs[0], projection=pc)
    dlat = half_width_km/111.19; dlon = dlat/np.cos(np.radians(ev.latitude))
    ax.set_extent([ev.longitude - dlon, ev.longitude + dlon, ev.latitude - 0.8*dlat, ev.latitude + 0.8*dlat], crs=pc)
    ax.add_feature(cfeature.LAND, alpha=0.35); ax.add_feature(cfeature.OCEAN, alpha=0.35)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4); gl.top_labels = gl.right_labels = False
    add_faults(ax, transform=pc)
    ax.scatter(land.longitude, land.latitude, marker="^", s=70, c="0.55", ec="k", transform=pc, zorder=4, label="land stations")
    for r in land.itertuples(): ax.text(r.longitude, r.latitude + 0.015, r.station, fontsize=7, ha="center", transform=pc, clip_on=True)
    ax.scatter(obs.longitude, obs.latitude, marker="s", s=45, c="C0", ec="k", transform=pc, zorder=5, label="ocean-bottom stations (OBS)")
    for r in obs.itertuples(): ax.text(r.longitude, r.latitude - 0.015, r.station[3:], fontsize=7, color="C0", ha="center", va="top", transform=pc)
    star(ax, ev.latitude, ev.longitude, size=300, transform=pc)
    ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
    ax.set_title(f"{ev_id}: the amphibious network (numbers: OBS)", fontsize=10)
    axs = fig.add_subplot(gs[1])
    sphere_base(axs, f"{ev_id}: focal sphere of the published mechanism" + (f" {sdr[0]:.0f}/{sdr[1]:.0f}/{sdr[2]:.0f}" if sdr else "")
                + "\n(red = compression: first motion up; white = dilatation: down)")
    if sdr: _beachball_background(axs, sdr)
    xs, ys = sphere_xy(land.azimuth, 180 - land.takeoff)
    axs.scatter(xs, ys, s=18, c="0.45", zorder=3, label=f"land stations ({len(land)})")
    xo, yo = sphere_xy(obs.azimuth, 180 - obs.takeoff)
    for k, r in obs.iterrows():
        sym = dict(marker="x", s=60) if r.p_polarity == 0 else dict(marker="^" if r.p_polarity > 0 else "v", s=90, ec="k")
        axs.scatter(xo[k], yo[k], c="C0", zorder=5, **sym)
        axs.text(xo[k] + 0.04, yo[k] + 0.04, r.station[3:], color="C0", fontsize=7.5, fontweight="bold", zorder=6)
    axs.scatter([], [], marker="^", c="C0", ec="k", label="OBS (▲ up, ▽ down, × near a nodal plane)")
    axs.legend(loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=1, fontsize=8, frameon=False)
    plt.show()
    print(f"{ev_id}: land stations {land.dist_km.min():.0f}–{land.dist_km.max():.0f} km, take-off {land.takeoff.min():.0f}–{land.takeoff.max():.0f}°;  "
          f"OBS {obs.dist_km.min():.0f}–{obs.dist_km.max():.0f} km, take-off {obs.takeoff.min():.0f}–{obs.takeoff.max():.0f}°;  "
          f"OBS first motions: {(obs.p_polarity > 0).sum()} up, {(obs.p_polarity < 0).sum()} down, "
          f"{(obs.p_polarity == 0).sum()} near a nodal plane")

In [ ]:
for ev_id in EVENT_IDS:
    show_amphibious(ev_id)

### 2.3 Choose the stations: signal-to-noise ratio

Not every station is worth picking. If the P wave is hardly above the noise, the first motion cannot be read reliably, and a wrong polarity hurts the solution more than a missing one. For every station we measure **S/N = largest amplitude in the first 0.15 s after the catalog P pick ÷ standard deviation of the noise in the 1.5 s before it** (vertical component, band-pass 1–15 Hz, causal), and drop the stations below `snr_min`.

Why only 0.15 s? We want to know whether the **first motion** stands out of the noise, not whether the station recorded a big P wave. At 100–150 km the first arrival is often a weak wave refracted at depth, followed a few tenths of a second later by a much stronger one: with a long window such a station gets a high S/N although its first motion is hardly visible. (Try `snr_window=0.5` and compare the maps.)

`select_stations(event_id, snr_min=10)` writes the selected stations to `skhash/<event_id>/pol.csv` (the file the picker and SKHASH use). The complete list of stations is kept in `pol_all.csv`, so you can run it again with other values at any time; polarities you have already picked are kept.

| Option | Default | Meaning |
|---|---|---|
| `snr_min` | 10 | minimum S/N (raise it to 15–20 for a stricter selection) |
| `snr_window` | 0.15 | length (s) of the window after the P pick in which the signal amplitude is measured |
| `min_sep_deg` | `None` | optional: drop any station whose ray leaves the source less than this angle from the ray of a better station (near-duplicates on the focal sphere, e.g. ISIL and SILT), e.g. `3` |
| `thin`, `min_spacing_km` | `{}`, 5 | optional: keep only the N best stations of a dense network, e.g. `thin={"KO": 10}` |

The map shows the kept stations colored by S/N and the dropped ones in gray. The printout also compares the catalog P pick with an automatic onset picker (AIC, Maeda 1985): the number of kept stations where both agree within 0.1 s.

In [ ]:
SELECTION = {ev_id: select_stations(ev_id, snr_min=10) for ev_id in EVENT_IDS}

### 2.4 Pick the P polarities, station by station

Pick the **first motion of the P wave on the vertical component**:

- **`u`** → `+1` (up, compression): the first deflection after the P arrival goes **up**.
- **`d`** → `-1` (down, dilatation): it goes **down**.
- **`0`** → `0` (unclear): noisy, emergent onset, no clear P. SKHASH ignores these stations.

`pick_polarities("<event_id>")` loops over the stations selected in 2.3 (closest first). For each station it shows:

| Panel | What to look at |
|---|---|
| top left | the **three components** (vertical in black, horizontals in gray, same scale) from 3 s before the P arrival to a few seconds after the S arrival. The dashed lines are the **catalog picks** (P red, S blue), made automatically by PhaseNet: the P onset should be at (or very close to) the red line |
| bottom left | **zoom on the P onset of the vertical component**: filtered trace (black) and raw trace (gray). The shaded band is the pre-P noise level: a first motion inside the band is not reliable |
| right | the focal sphere with all the stations of the event and **your picks so far** (black ▲ up, white ▽ down, gray × unclear, small gray dots: not picked yet). The station you are picking is circled in red |

Then answer with the **buttons** above the figure:

| Button (typed key) | Action |
|---|---|
| **▲ up** / **▼ down** / **? unclear** (`u` / `d` / `0`) | up / down / unclear (saved immediately in `skhash/<event_id>/pol.csv`), then the next station |
| **keep / skip** (Enter) | keep the current value (unclear for a station not picked yet) and go to the next station |
| **↕ vertical zoom** (`z`) | zoom in vertically on the onset (×3, ×10, back to ×1) when the first motion is small compared with what follows |
| **↔ wider window** (`w`) | wider time window in the zoom panel (−0.5 to +1 s, click again to go back): to see what comes after an emergent onset |
| **◀ back** (`b`) | go back to the previous station |
| **■ stop** (`q`) | stop (everything you picked is already saved; run the cell again to continue where you stopped) |

The buttons need `ipywidgets` (part of the course environment). Without it, or with `pick_polarities(..., buttons=False)`, the notebook asks you to type the key in brackets and press Enter.

Useful options: `order="azimuth"` (go around the epicenter: watch the polarities flip from one quadrant to the next), `max_dist_km=100` (only the closest stations), `stations=["ELBA", "BAND"]` (re-check a few stations), `redo=True` (go through all the stations again).

> **What real onsets look like.** Close stations (up to ~70 km) usually show a sharp first motion right at the red line. Farther away the first arrival is a wave refracted in the lower crust or at the Moho: it is weaker and more emergent, and a stronger arrival often follows 0.1–0.3 s later. Read the motion of the **first** arrival, not of the larger one behind it (**↔ wider window** and **↕ vertical zoom** help). The catalog picks can be a few hundredths of a second early or late: read the first motion where the trace leaves the noise band. If you hesitate, answer **? unclear**: a missing polarity costs little, a wrong one can rotate the solution.

> **Why a causal filter?** The traces are filtered with a *causal* (one-way) band-pass filter. A *zero-phase* filter (`zerophase=True`) is symmetric in time and can create small oscillations **before** the true onset, which can fool you into picking the wrong polarity. See it for yourself with `compare_filters(...)` below the picker.

> **Raw data.** The waveforms are raw counts (no instrument correction): this does not matter for the polarity, as long as the sensors are connected with the standard polarity (up = positive). A station whose picks disagree with the solution in every event may have a reversed sensor.

In [ ]:
# Pick the polarities of one event. Run the cell again to continue where you stopped.
# Then change the event: EVENT_IDS[1], EVENT_IDS[2]  (or split the three events among groups)
pick_polarities(EVENT_IDS[0])
# No buttons, only a line of text like VBox(...)? Then ipywidgets does not work in your Jupyter: use
# pick_polarities(EVENT_IDS[0], buttons=False) and type the answers instead.
# show_station(EVENT_IDS[0], "ELBA")    # look at one station without picking

**Optional — why a causal filter?** The same onset without filter (gray), with a causal band-pass (black) and with a zero-phase band-pass of the same band and order (red). Look just *before* the red line: which trace starts moving first, and in which direction? Try other stations of the event.

In [ ]:
compare_filters(EVENT_IDS[0], "ELBA")

### 2.5 Map your picks

`plot_polarity_map("<event_id>")` shows the polarities of a run on a map (left) and on the focal sphere (right): black ▲ = up, white ▽ = down, gray × = unclear / not picked. The title gives the number of stations with a polarity, their distance range and the **azimuthal gap** (largest angle around the epicenter without a picked station).

#### Exercise 4 — Before running SKHASH, look at the figure

- Can you already see where the ups and the downs are? Draw (mentally) two perpendicular planes that separate them on the sphere.
- Where do the stations fall on the sphere? The events are 10–14 km deep. Close stations receive a **direct, upgoing** ray (take-off < 90°): it is plotted on the lower hemisphere at the *opposite* azimuth. Farther away the first P is a wave refracted in the crust (leaving the source slightly downwards, near the rim) or at the Moho (> ~125 km, leaving more steeply downwards: further inside). Can you find each group on the sphere?
- Are there stations with a polarity that does not fit its neighbors? Re-check them: `pick_polarities("<event_id>", stations=["XXXX"])`.

In [ ]:
plot_polarity_map(EVENT_IDS[0])            # labels=True writes the station names

### 2.6 The polarity file SKHASH reads

The picker wrote your answers in `skhash/<event_id>/pol.csv` (inside `focal_stress_data/data/`), **one line per selected station** (`pol_all.csv`, next to it, lists all the stations with data). The take-off angles and azimuths were computed beforehand with the 1D velocity model (`make_skhash_2025.py`); only the column `p_polarity` comes from you (the value in the example is illustrative). You can open the file with a plain text editor (or Jupyter's file browser) and check it, e.g.:

```
event_id, station, network, location, channel, p_polarity, takeoff, takeoff_uncertainty, azimuth, azimuth_uncertainty
2815, ELBA, TU, 00, HHZ, 1.0, 81.4, 6.5, 26.2, 1.9
```

| Column | Meaning |
|---|---|
| `p_polarity` | `1.0` = first motion **up** (compression), `-1.0` = **down** (dilatation), `0.0` = unclear / not picked (the station is ignored) |
| `takeoff` | angle at which the ray leaves the source, from the vertical: 0° = straight up, 90° = horizontal, 180° = straight down |
| `azimuth` | direction from the epicenter to the station (° clockwise from north) |
| `*_uncertainty` | uncertainty of these angles (from the location uncertainty), used by SKHASH in its Monte Carlo trials |

You can still edit `p_polarity` by hand: do not change the other columns, their order or the commas, and do not add header or comment lines. To **leave a station out**, set its `p_polarity` to `0.0` or delete its line (or change the selection: `select_stations(..., snr_min=...)`). (The picker also keeps `picking_log.csv`, the list of stations you have already seen; delete it to start picking from scratch.)

### 2.7 Run SKHASH

`run_skhash(folder)` runs SKHASH in the folder `skhash/<folder>/` (it uses the `eq_catalog.csv`, `pol.csv` and `control_file.txt` of that folder) and writes the results there (`out_*_mechanism.csv`, `out_*_acceptable_planes.csv`, …). `folder` is an event id, or the name of a subset folder (section 2.9).

Main control parameters (in `control_file.txt` of each folder):

| Parameter | Meaning |
|---|---|
| `npolmin` | minimum number of polarities needed to compute the mechanism |
| `max_agap` | maximum **azimuthal gap** allowed (°). If the stations leave a larger gap, the event is skipped |
| `max_pgap` | maximum **take-off angle gap** allowed (°, max 90 in SKHASH). Shallow earthquakes recorded at regional distances have most rays near-horizontal, so this gap is usually large |
| `badfrac`, `badmin` | fraction / minimum number of polarities assumed to be wrong (tolerates picking errors) |
| `nmc` | number of Monte Carlo trials that perturb the location, take-off angles and azimuths |
| `maxout` | maximum number of acceptable solutions kept |
| `dang` | grid spacing (°) of the search over strike, dip and rake |

> **The velocity model.** The take-off angles in `pol.csv` were computed by `make_skhash_2025.py` with the 1D Marmara model written at the top of that script (`VEL_MODEL`); it reproduces the P travel times predicted by NonLinLoc for the catalog picks to within ~0.1 s. Re-running that script with `OVERWRITE_POL = True` recomputes the angles **and resets all the picks to 0**.
>
> **Class settings vs. the published catalog.** Here: unweighted polarities only, `npolmin` 6, `max_agap` 180°, `max_pgap` 90°. The catalog used weighted polarities read by a neural network plus S/P amplitude ratios, at least 8 polarities, `max_agap` 120° and `max_pgap` 60°.

In [ ]:
def _param(folder, name, default):
    lines = [l.strip() for l in (folder/"control_file.txt").read_text().splitlines() if l.strip()]
    return float(lines[lines.index(f"${name}") + 1]) if f"${name}" in lines else default

def run_skhash(event_folder):
    # event_folder: name of a folder inside skhash/, e.g. "20190925160003" or "20190925160003_within70km"
    folder = SKHASH_DIR/str(event_folder)
    for f in folder.glob("out_*.csv"): f.unlink()                   # remove old results (never show a stale solution)
    pol = pd.read_csv(folder/"pol.csv", skipinitialspace=True)
    n_pol = int((pol.p_polarity.fillna(0).abs() > 0).sum()); npolmin = _param(folder, "npolmin", 8)
    print(f"{event_folder}: {n_pol} polarities (npolmin = {npolmin:g})")
    if n_pol < npolmin:
        print("   -> not enough polarities: pick more stations (pick_polarities) or lower npolmin in control_file.txt."); return
    try:
        import SKHASH                                               # is it installed in THIS kernel's Python?
    except ImportError:
        print(f"SKHASH is not installed in the Python used by this notebook ({sys.executable}).\n"
              "Select the kernel of the course environment (see README.md)."); return
    code = "import sys; from SKHASH.SKHASH import main; sys.argv = ['SKHASH', 'control_file.txt']; sys.exit(main())"
    res = subprocess.run([sys.executable, "-c", code], cwd=folder, capture_output=True, text=True)
    for l in res.stdout.splitlines():
        if any(k in l for k in ("Skipping", "Discarded", "WARNING", "Ignoring")): print("   SKHASH:", l.strip())
    if res.returncode != 0 and not list(folder.glob("out_*_mechanism.csv")):
        print("   SKHASH error:", res.stderr.strip().splitlines()[-1] if res.stderr.strip() else "unknown")
    out = next(folder.glob("out_*_mechanism.csv"), None)
    if out is None or len(pd.read_csv(out)) == 0:
        print("   -> no mechanism was produced (see the SKHASH message above: gap too large? too few polarities?)")
    else:
        print("   -> done:", out.name)

for ev_id in EVENT_IDS:          # all stations with a polarity, for the three events
    run_skhash(ev_id)

**Check: does SKHASH read the take-off angles as we write them?** (Optional, 1 minute.) Conventions differ between programs (2.1). `check_takeoff_convention` invents the polarities of a known, **oblique** mechanism at the stations of an event, computed with our convention (take-off 0° = up), and runs SKHASH twice: with the take-off angles as written in `pol.csv`, and mirrored (180° − take-off). SKHASH must recover the mechanism in the first case only. Only synthetic polarities are used: your picks are not touched.

In [ ]:
def check_takeoff_convention(ev_id, sdr=(30, 50, 60)):
    """Synthetic polarities of the mechanism sdr (strike, dip, rake) at the stations of an event, run through SKHASH
    with the take-off angles as written and mirrored; returns the Kagan angle to sdr of both solutions."""
    ev_id = str(ev_id); src = SKHASH_DIR/ev_id
    pol = read_pol(ev_id, "pol_all.csv" if (src/"pol_all.csv").exists() else "pol.csv")
    rad = radiation(sdr, pol.azimuth, pol.takeoff)
    pol["p_polarity"] = np.where(np.abs(rad) < 0.1, 0.0, np.sign(rad))
    rows = []
    for name, mirrored in (("test_takeoff_as_written", False), ("test_takeoff_mirrored", True)):
        dst = SKHASH_DIR/f"{ev_id}_{name}"; dst.mkdir(exist_ok=True)
        for f in ("eq_catalog.csv", "control_file.txt"): shutil.copy(src/f, dst/f)
        p = pol.copy()
        if mirrored: p["takeoff"] = (180 - p["takeoff"]).round(1)
        write_pol(dst.name, p); run_skhash(dst.name)
        out = next(dst.glob("out_*_mechanism.csv"), None)
        m = pd.read_csv(out).iloc[0] if out is not None and len(pd.read_csv(out)) else None
        rows.append({"take-off angles given to SKHASH": "as written (0° = up)" if not mirrored else "mirrored (0° = down)",
                     "SKHASH solution": "-" if m is None else f"{m.strike:.0f}/{m.dip:.0f}/{m.rake:.0f}",
                     "Kagan angle to the true mechanism (°)": None if m is None else round(kagan_angle((m.strike, m.dip, m.rake), sdr))})
    print(f"true mechanism {sdr[0]}/{sdr[1]}/{sdr[2]}, stations of event {ev_id}:")
    display(pd.DataFrame(rows).set_index("take-off angles given to SKHASH"))

check_takeoff_convention(EVENT_IDS[0])

### 2.8 Look at the results

For each run: your preferred mechanism (red nodal planes), the **published solution** of the catalog (dashed black, with the Kagan angle between the two printed above the figure), **all acceptable solutions** (light-blue planes — their spread is the uncertainty), and the stations on the focal sphere (black = U, white = D; red edge = station that **disagrees** with the solution — re-check that pick with the picker, the command is printed below the figure). Quality grades go from **A** (best) to **D**.

Take-off angles are plotted from the downward vertical (lower hemisphere, equal area).

**Right: the geological check.** A map around the epicenter with the mapped fault traces (thin red lines), your mechanism (beach ball) and the **strike of its two nodal planes** drawn through the epicenter (red; published solution: black dashed; labels: strike/dip). Is one of the nodal planes parallel to the mapped faults? Then that plane is the more likely fault plane. Careful: a dipping plane reaches the surface away from the epicenter (about depth / tan(dip) towards its up-dip side), so compare **directions**, not positions.

In [ ]:
from obspy.imaging.beachball import aux_plane       # sphere_base, sphere_xy and nodal_xy: see the tools cell of Part 2

def _catalogue_solution(ev_id):
    """Published mechanism of an event (mechanisms_catalog.csv), or None."""
    if not CATALOG_CSV.exists(): return None
    cat = pd.read_csv(CATALOG_CSV, dtype={"event_id": str})
    c = cat[cat.event_id.str.strip() == str(ev_id)]
    return None if c.empty else c.iloc[0]

def _solution_map(ax, ev_id, sdr, ref=None, half_width_km=15):
    """Map around the epicenter: mapped faults, and the strike of the two nodal planes of a solution (and of the
    published one) drawn through the epicenter."""
    ev = EVENTS.loc[str(ev_id)]; pc = ccrs.PlateCarree()
    dlat = half_width_km/111.19; dlon = dlat/np.cos(np.radians(ev.latitude))
    ax.set_extent([ev.longitude - dlon, ev.longitude + dlon, ev.latitude - dlat, ev.latitude + dlat], crs=pc)
    ax.add_feature(cfeature.LAND, alpha=0.35); ax.add_feature(cfeature.OCEAN, alpha=0.35)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4); gl.top_labels = gl.right_labels = False
    add_faults(ax, transform=pc)
    for mech, c, ls, lab in ((ref, "k", "--", "published"), (sdr, "r", "-", "your solution")):
        if mech is None: continue
        s2, d2, _ = aux_plane(*mech)
        for k, (st, dp) in enumerate(((mech[0], mech[1]), (s2, d2))):
            la1, lo1 = offset_point(ev.latitude, ev.longitude, 0.72*half_width_km, st)
            la2, lo2 = offset_point(ev.latitude, ev.longitude, 0.72*half_width_km, st + 180)
            ax.plot([lo2, lo1], [la2, la1], color=c, ls=ls, lw=1.6, transform=pc, zorder=4,
                    label=f"{lab}: strike of the nodal planes" if k == 0 else None)
            ax.text(lo1, la1, f" {st % 360:.0f}°/{dp:.0f}°", color=c, fontsize=7.5, transform=pc, zorder=5)
    ax.add_collection(beach(list(sdr), xy=(ev.longitude, ev.latitude), width=40, linewidth=0.8, facecolor="r",
                            axes=ax, zorder=6))
    ax.legend(loc="lower left", fontsize=7, framealpha=0.9)
    ax.set_title("nodal planes and mapped faults (labels: strike/dip)", fontsize=9)

def show_result(event_folder, n_planes=150):
    folder = SKHASH_DIR/str(event_folder)
    out = next(folder.glob("out_*_mechanism.csv"), None)
    if out is None or len(pd.read_csv(out)) == 0: print(f"{event_folder}: no solution to show. Run run_skhash first."); return
    m = pd.read_csv(out).iloc[0]
    acc = pd.read_csv(next(folder.glob("out_*_acceptable_planes.csv"))); info = pd.read_csv(next(folder.glob("out_*_pol_info.csv")))
    print(f"{event_folder}:  strike/dip/rake = {m.strike:.0f}/{m.dip:.0f}/{m.rake:.0f}   quality = {m.quality}   "
          f"nodal-plane uncertainty = {m.fault_plane_uncertainty:.0f}° / {m.aux_plane_uncertainty:.0f}°   polarities = {int(m.num_p_pol)}   "
          f"misfit = {m.polarity_misfit:.1f}%   azimuthal gap = {m.azimuthal_gap:.0f}°   take-off gap = {m.takeoff_gap:.0f}°   acceptable solutions = {len(acc)}")
    fig = plt.figure(figsize=(12, 5.6)); gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.15], wspace=0.1)
    ax = fig.add_subplot(gs[0]); sphere_base(ax, f"{event_folder}: focal sphere")
    for r in acc.sample(min(n_planes, len(acc)), random_state=0).itertuples():
        s2, d2, _ = aux_plane(r.strike, r.dip, r.rake)
        for ss, dd in [(r.strike, r.dip), (s2, d2)]:
            x, y = nodal_xy(ss, dd); ax.plot(x, y, color="C0", lw=0.4, alpha=0.25, zorder=1)
    s2, d2, _ = aux_plane(m.strike, m.dip, m.rake)
    for k, (ss, dd) in enumerate([(m.strike, m.dip), (s2, d2)]):
        x, y = nodal_xy(ss, dd); ax.plot(x, y, "r", lw=2, zorder=2, label="your solution" if k == 0 else None)
    ref = _catalogue_solution(event_of(event_folder))                   # published solution, if any
    if ref is not None:
        s2, d2, _ = aux_plane(ref.strike, ref.dip, ref.rake)
        for k, (ss, dd) in enumerate([(ref.strike, ref.dip), (s2, d2)]):
            x, y = nodal_xy(ss, dd); ax.plot(x, y, "k--", lw=1.4, zorder=2, label="published (catalog)" if k == 0 else None)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2, fontsize=7.5, frameon=False)
        print(f"   published solution: {ref.strike:.0f}/{ref.dip:.0f}/{ref.rake:.0f} (quality {ref.quality}); Kagan angle between "
              f"your solution and the published one: {kagan_angle((m.strike, m.dip, m.rake), (ref.strike, ref.dip, ref.rake)):.0f}°")
    x, y = sphere_xy(info.azimuth, 180 - info.takeoff)                  # SKHASH takeoff: 0 = up, 180 = down
    ok = (info.pol_agreement.astype(int) == 1) if "pol_agreement" in info.columns else pd.Series(True, index=info.index)
    for sign, fc in [(1, "k"), (-1, "w")]:
        mm = np.sign(info.p_polarity) == sign
        ax.scatter(x[mm & ok], y[mm & ok], s=40, c=fc, ec="k", zorder=3); ax.scatter(x[mm & ~ok], y[mm & ~ok], s=50, c=fc, ec="red", lw=2, zorder=4)
    _solution_map(fig.add_subplot(gs[1], projection=ccrs.PlateCarree()), event_of(event_folder), (m.strike, m.dip, m.rake),
                  None if ref is None else (ref.strike, ref.dip, ref.rake))
    plt.show()
    bad = info[~ok]
    if len(bad):
        col = "sta_code" if "sta_code" in bad.columns else "station"
        names = [c.split(".")[1] if c.count(".") >= 1 else c for c in bad[col].astype(str)]   # NET.STA.LOC.CHA -> STA
        virt = [n for n in names if n.startswith("OBS")]                                         # ocean-bottom stations of 2.2
        real = [n for n in names if n not in virt]
        if real:
            print("Stations that DISAGREE with the solution (re-check these polarities):", ", ".join(real))
            print(f"   -> re-check them with:  pick_polarities('{event_of(event_folder)}', stations={real})")
        if virt:
            print("OBS that disagree with the solution (their polarity comes from the published mechanism):", ", ".join(virt))

for ev_id in EVENT_IDS:
    show_result(ev_id)

### 2.9 Change the stations: distance, azimuthal coverage, number of stations

How much does the mechanism depend on **which** stations you use? `make_subset(event_id, name, ...)` creates a new folder `skhash/<event_id>_<name>/` with the same event and control file but only some of the stations (with the polarities you picked). You can then run it, map it and compare it like any other folder.

| Option of `make_subset` | Keeps | What it tests |
|---|---|---|
| `max_dist_km=70` | stations closer than 70 km | few, close stations: larger azimuthal gap? |
| `min_dist_km=100` | stations farther than 100 km | distant stations: refracted waves only; beyond ~125 km the first P is refracted at the Moho and leaves the source more steeply downwards |
| `az_range=(270, 90)` | stations with azimuth between 270° and 90°, i.e. north of the epicenter (`(90, 270)`: south) | many stations, but all on one side of the epicenter |
| `n_closest=10` | the 10 closest stations with a polarity | number of stations |
| `n_random=10, seed=1` | 10 random stations with a polarity (change `seed` for another draw) | number of stations, by chance |
| `obs=OBS_STATIONS` | adds the 12 ocean-bottom stations of 2.2, with the polarity predicted by the published mechanism (as if they had been picked without error) | what would an amphibious network add? |
| `stations=[...]` / `exclude=[...]` | only / all but these stations | your own choice |

Options can be combined (e.g. `min_dist_km=100, az_range=(0, 180)`). Any other keyword changes the SKHASH control file of the subset, e.g. `max_agap=300` or `npolmin=4`. `compare_runs([...])` puts the solutions side by side: number of polarities, distance range, azimuthal and take-off gaps, quality, nodal-plane uncertainty, misfit, and the **Kagan angle** with respect to the first run in the list (the smallest 3-D rotation that turns one mechanism into the other: 0° = identical, 120° = the largest possible difference). The published solution of the catalog (`mechanisms_catalog.csv`) is added as a reference.

> If you change picks in `skhash/<event_id>/pol.csv` after creating a subset, create the subset again: it keeps a copy of the polarities it was made with.

In [ ]:
# --- helper code for station subsets and comparisons: just run this cell ---
def _set_params(ctl_file, params):
    """Change (or add) $parameters of a SKHASH control file, e.g. max_agap=300."""
    lines = ctl_file.read_text().splitlines(); keys = [l.strip() for l in lines]
    for k, v in params.items():
        if f"${k}" in keys: lines[keys.index(f"${k}") + 1] = str(v)
        else: lines += [f"${k}", str(v)]
    ctl_file.write_text("\n".join(lines) + "\n")

def make_subset(event_id, name, max_dist_km=None, min_dist_km=None, az_range=None, n_closest=None, n_random=None,
                seed=0, stations=None, exclude=None, obs=None, **skhash_params):
    """Create skhash/<event_id>_<name>/ with only some of the stations of skhash/<event_id>/ (picks included).
    obs: ocean-bottom stations to add (OBS_STATIONS of 2.2), with the polarity predicted by the published mechanism.
    Extra keyword arguments change the SKHASH control file of the subset, e.g. max_agap=300 or npolmin=4."""
    event_id = str(event_id); src = SKHASH_DIR/event_id
    pol = pol_geometry(event_id); keep = pd.Series(True, index=pol.index)
    if max_dist_km is not None: keep &= pol.dist_km <= max_dist_km
    if min_dist_km is not None: keep &= pol.dist_km >= min_dist_km
    if az_range is not None:
        a0, a1, az = az_range[0] % 360, az_range[1] % 360, pol.azimuth % 360
        keep &= ((az >= a0) & (az <= a1)) if a0 <= a1 else ((az >= a0) | (az <= a1))   # (300, 60) wraps through north
    if stations is not None: keep &= pol.station.isin([s.upper() for s in stations])
    if exclude is not None: keep &= ~pol.station.isin([s.upper() for s in exclude])
    sub = pol[keep]
    picked = sub[sub.p_polarity != 0]
    if n_closest is not None: sub = picked.nsmallest(n_closest, "dist_km")
    elif n_random is not None: sub = picked.sample(min(n_random, len(picked)), random_state=seed)
    sub = sub.sort_index()
    if obs is not None and len(obs):                                             # ocean-bottom stations of 2.2
        sub = pd.concat([sub, obs_pol_rows(event_id, obs)], ignore_index=True)
    dst = SKHASH_DIR/f"{event_id}_{name}"; dst.mkdir(exist_ok=True)
    for f in ("eq_catalog.csv", "control_file.txt"): shutil.copy(src/f, dst/f)
    if skhash_params: _set_params(dst/"control_file.txt", skhash_params)
    for f in dst.glob("out_*.csv"): f.unlink()                                    # results of an older subset
    write_pol(dst.name, sub)
    p = sub[sub.p_polarity != 0]
    if len(p):
        print(f"{dst.name}: {len(p)} stations with a polarity, {p.dist_km.min():.0f}–{p.dist_km.max():.0f} km, "
              f"azimuthal gap {azimuthal_gap(p.azimuth):.0f}°, take-off {p.takeoff.min():.0f}–{p.takeoff.max():.0f}°")
    else:
        print(f"{dst.name}: no station with a polarity (did you pick event {event_id} first?)")
    if skhash_params: print("   control file changed:", ", ".join(f"${k} = {v}" for k, v in skhash_params.items()))
    if obs is not None and len(obs):
        v = sub[sub.network == "OB"]
        print(f"   + {len(v)} OBS ({(v.p_polarity != 0).sum()} with a readable polarity), {v.dist_km.min():.0f}–{v.dist_km.max():.0f} km, "
              f"take-off {v.takeoff.min():.0f}–{v.takeoff.max():.0f}°")
    return dst.name

def _solution(folder):
    out = next((SKHASH_DIR/str(folder)).glob("out_*_mechanism.csv"), None)
    if out is None: return None
    m = pd.read_csv(out)
    return None if len(m) == 0 else m.iloc[0]

def compare_runs(folders):
    """Table and beach balls of several SKHASH runs (folders in skhash/). Kagan angles are relative to the first run with a solution."""
    rows, ev_id = [], event_of(folders[0])
    for f in map(str, folders):
        pol = pol_geometry(f); p = pol[pol.p_polarity != 0]; m = _solution(f)
        name = "all picks" if f == ev_id else f.replace(f"{ev_id}_", "")
        row = dict(run=name, n_pol=len(p), dist_km=f"{p.dist_km.min():.0f}–{p.dist_km.max():.0f}" if len(p) else "-",
                   az_gap=round(azimuthal_gap(p.azimuth)) if len(p) else None)
        if m is not None:
            row.update(strike=m.strike, dip=m.dip, rake=m.rake, quality=m.quality, takeoff_gap=round(m.takeoff_gap),
                       plane_unc=round(m.fault_plane_uncertainty), misfit_pct=round(m.polarity_misfit, 1))
        rows.append(row)
    c = _catalogue_solution(ev_id)                                                # published solution, if any
    if c is not None:
        rows.append(dict(run="published (catalog)", n_pol=int(c.num_p_pol), dist_km="-", strike=c.strike, dip=c.dip, rake=c.rake,
                         quality=c.quality, az_gap=round(c.azimuthal_gap), takeoff_gap=round(c.takeoff_gap),
                         plane_unc=round(c.fault_plane_uncertainty), misfit_pct=round(c.polarity_misfit, 1)))
    df = pd.DataFrame(rows)
    ref = next((r for r in rows if "strike" in r), None)
    df["kagan_deg"] = [round(kagan_angle((ref["strike"], ref["dip"], ref["rake"]), (r["strike"], r["dip"], r["rake"])))
                       if (ref is not None and "strike" in r) else None for r in rows]
    print(f"Event {ev_id}: SKHASH solutions (Kagan angle relative to '{ref['run'] if ref else '-'}')")
    display(df.set_index("run"))
    fig, axs = plt.subplots(1, len(rows), figsize=(2.3*len(rows), 2.9), squeeze=False)
    for ax, r in zip(axs[0], rows):
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
        if "strike" in r:
            ax.add_collection(beach([r["strike"], r["dip"], r["rake"]], xy=(0, 0), width=2, facecolor="k", linewidth=1))
            sub = f"{r['strike']:.0f}/{r['dip']:.0f}/{r['rake']:.0f}  ({r['quality']})"
        else:
            ax.text(0, 0, "no solution", ha="center", va="center", color="0.4"); sub = ""
        ax.set_title(f"{r['run']}\n{sub}", fontsize=7.5)
    plt.tight_layout(); plt.show()

In [ ]:
# Build some subsets of one event (it must be picked first) and run SKHASH on each of them
EV = EVENT_IDS[0]
RUNS = [EV,                                                       # all the stations you picked
        make_subset(EV, "within70km", max_dist_km=70),            # only close stations
        make_subset(EV, "beyond100km", min_dist_km=100),          # only distant stations
        make_subset(EV, "north_only", az_range=(270, 90)),        # many stations, but only north of the epicenter
        make_subset(EV, "random10", n_random=10, seed=1),         # 10 random stations
        make_subset(EV, "all_plus_OBS", obs=OBS_STATIONS),        # your picks + the 12 ocean-bottom stations of 2.2
        make_subset(EV, "within70km_plus_OBS", max_dist_km=70, obs=OBS_STATIONS)]   # close land stations + OBS
for f in RUNS[1:]:
    run_skhash(f)

In [ ]:
# The working subsets: which stations (and which polarities) are left in each run?
for f in RUNS[1:]:
    plot_polarity_map(f)

In [ ]:
for f in RUNS[1:]:
    show_result(f)          # spread of the acceptable solutions of each subset
compare_runs(RUNS)

#### Exercise 5 — Discussion

- Section 2.3 dropped the stations with a low S/N. Is a polarity you are unsure of better than no polarity? How many wrong polarities does SKHASH tolerate (`badfrac`)?
- Which event gets the best quality grade? What limits the quality of the others: number of polarities, azimuthal gap, **take-off gap**?
- What happens when you use only a few stations? Does the *preferred* mechanism change, or mostly the **spread of acceptable solutions** (nodal-plane uncertainty)?
- Close stations vs. distant stations: which subset gives the more stable solution? Look at the take-off range printed by `make_subset`: which rays leave the source upwards, which downwards?
- `north_only` has many stations but they are all on one side. Did SKHASH accept it (`max_agap`)? If it was skipped, make it again allowing a larger gap, `make_subset(EV, "north_only_gap300", az_range=(270, 90), max_agap=300)`, run it and judge the result. Is the number of stations or their distribution more important?
- Is a Kagan angle of 30° between two solutions a big difference? Compare it with the nodal-plane uncertainty that SKHASH gives.
- Which stations disagree with the solution? Are they near a nodal plane, noisy, or did you mis-pick them? Does the same station disagree in all three events (a reversed sensor?).
- Most rays leave the source close to horizontal and plot near the rim of the focal sphere. What would help to constrain the dip of the planes (closer stations, ocean-bottom stations above the source, S/P amplitude ratios as in the published catalog)?
- Compare your solutions with the published ones. Where do they differ, and why could that be (number of polarities, weights, S/P ratios, your picks)?

## Part 3 — Stress field from focal mechanisms

A focal mechanism tells us how **one** fault slipped. A **population** of focal mechanisms tells us about the **stress** that made all of them slip. In this part you will:

1. Load the catalog of focal mechanisms of the **2019 Mw 5.8 Silivri sequence** and map it together with the mapped faults.
2. Look at the P and T axes and the faulting types, and **predict** the stress field.
3. See what the stress inversion needs (**inputs**) and what it gives back (**outputs**).
4. Run **StressInverse** (Vavryčuk, 2014) step by step and interpret the result.
5. Check whether the result is **reliable** and how much it changes when you change the inputs.
6. See **which earthquakes fit** the stress, map S<sub>Hmax</sub> and S<sub>hmin</sub>, and ask which nodal plane is the fault.

**Why the 2019 sequence?** The Silivri earthquake of 26 September 2019 (Mw 5.8) ruptured a blind thrust fault in the Kumburgaz Basin, a few kilometers north-west of the 2025 mainshock (Durand et al., 2020; Karabulut et al., 2020), with a complex source that combined thrust and strike-slip faulting (Karabulut et al., 2020; Turhan et al., 2023). Its 71 mechanisms come from a small volume around one rupture and have **diverse orientations** (strike-slip, normal and thrust components), which is what a stress inversion needs (see 3.2). Many of them have quality C: we will see what that means for the result.

**And the 2025 sequence?** The 2025 mainshock ruptured a different, although related, fault, and the two sequences barely overlap (compare the map below with the figure of Part 2). The 2019 earthquake also changed the stress around it: Coulomb stress models suggest that it brought the 2025 rupture area closer to failure (Karabulut et al., 2020; Şahin et al., 2022). The stress we find here is the stress of the 2019 volume, not necessarily the one that drove the 2025 sequence, so we do **not** test the 2025 mechanisms against it (see the discussion at the end).

Part 3 does not need the results of Parts 1 and 2: after a restart of the kernel, run the cells of *Prepare the workspace* (environment check, imports, download, setup) and then continue here.

The catalog `mechanisms_catalog_2019.csv` has one row per earthquake: strike, dip and rake of the preferred nodal plane, the SKHASH quality parameters (quality grade, nodal-plane uncertainty, number of polarities, gaps, misfit…), and the origin time, location and magnitude.

In [ ]:
# Path to the catalog (2019 Silivri sequence)
catalog_path = CATALOG_2019_CSV
# Read catalog
catalog = pd.read_csv(catalog_path)

print(f"Number of focal mechanisms: {len(catalog)}")
print(f"Number of variables: {len(catalog.columns)}")
print("Quality grades:", catalog.quality.value_counts().sort_index().to_dict())

catalog.head()

In [ ]:
# Geographic map of focal mechanisms (2019 Silivri sequence) with the mapped faults
fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=ccrs.PlateCarree())

# Geographic background
ax.add_feature(cfeature.LAND, alpha=0.4)
ax.add_feature(cfeature.OCEAN, alpha=0.3)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.gridlines(draw_labels=True, alpha=0.3)
add_faults(ax, transform=ccrs.PlateCarree(), linewidths=1.0)

# Plot focal mechanisms (largest first, so that the small ones stay visible on top)
for _, event in catalog.sort_values("magnitude", ascending=False).iterrows():
    width = 15 * 1.5**(event["magnitude"] - 2)              # beachball size scales with magnitude
    bball = beach([event["strike"], event["dip"], event["rake"]], xy=(event["origin_lon"], event["origin_lat"]),
                  width=width, linewidth=0.5, zorder=3, axes=ax)
    ax.add_collection(bball)
star(ax, MAINSHOCK_2019["lat"], MAINSHOCK_2019["lon"], MAINSHOCK_2019["label"], transform=ccrs.PlateCarree())
ax.scatter(MAINSHOCK["lon"], MAINSHOCK["lat"], marker="*", s=250, c="0.75", ec="k", zorder=10,
           transform=ccrs.PlateCarree(), label="2025 mainshock (Part 2)")

# Map extent: the catalog plus a margin
lat_c = catalog["origin_lat"].mean()
half = max(0.07, (catalog["origin_lat"].max() - catalog["origin_lat"].min())/2 + 0.03)
ax.set_extent([catalog["origin_lon"].min() - 0.08, catalog["origin_lon"].max() + 0.08, lat_c - half, lat_c + half],
              crs=ccrs.PlateCarree())

ax.legend(loc="lower left", fontsize=8)
ax.set_title(f"Focal mechanisms of the 2019 Silivri sequence ({len(catalog)}), mapped faults in red", fontsize=10)
plt.tight_layout()
plt.show()

### 3.1 Before the inversion: P and T axes, faulting types

Every mechanism has a **P axis** (in the middle of the dilatational quadrant, white in the beach ball) and a **T axis** (in the middle of the compressional quadrant, black). They are the directions of maximum shortening and extension *of that earthquake*, and they are **not** the principal stresses: a pre-existing fault can slip even if it is not optimally oriented, so σ₁ can lie anywhere in the dilatational quadrant of the mechanism (McKenzie, 1969), not necessarily at P. But when many mechanisms with **different** orientations are plotted together, the cloud of P axes surrounds σ₁ and the cloud of T axes surrounds σ₃.

The faulting type of each mechanism is classified from the plunges of its P, B (null) and T axes, with the scheme of the World Stress Map (Zoback, 1992):

| Class | Meaning | P axis plunge | B axis plunge | T axis plunge |
|---|---|---|---|---|
| NF | normal faulting | ≥ 52° | | ≤ 35° |
| NS | normal + strike-slip | 40°–52° | | ≤ 20° |
| SS | strike-slip | < 40° (or ≤ 20°) | ≥ 45° | ≤ 20° (or < 40°) |
| TS | thrust + strike-slip | ≤ 20° | | 40°–52° |
| TF | thrust faulting | ≤ 35° | | ≥ 52° |
| U | undefined (oblique) | any other combination | | |

In [ ]:
# --- helper code: P/B/T axes, faulting regime and stereonets (just run this cell) ---
REGIME_NAME = dict(NF="normal faulting", NS="normal + strike-slip (transtension)", SS="strike-slip",
                   TS="thrust + strike-slip (transpression)", TF="thrust faulting", U="undefined (oblique)")

def trend_plunge(v):
    """Trend (deg from north) and plunge (deg, downwards) of an axis given as a vector (north, east, down)."""
    v = np.real(np.asarray(v)).astype(float); v = v/np.linalg.norm(v)
    if v[2] < 0: v = -v                                                         # lower hemisphere
    return np.degrees(np.arctan2(v[1], v[0])) % 360, np.degrees(np.arcsin(np.clip(v[2], -1, 1)))

def pbt_axes(strike, dip, rake):
    """P, B and T axes (trend, plunge) of a double-couple mechanism."""
    n, u = fault_vectors(strike, dip, rake)
    p, t = (n - u)/np.sqrt(2), (n + u)/np.sqrt(2)
    return trend_plunge(p), trend_plunge(np.cross(n, u)), trend_plunge(t)

def wsm_regime(pl_p, pl_b, pl_t):
    """Faulting regime from the plunges of the P (or sigma1), B (sigma2) and T (sigma3) axes (Zoback, 1992)."""
    if pl_p >= 52 and pl_t <= 35: return "NF"
    if 40 <= pl_p < 52 and pl_t <= 20: return "NS"
    if (pl_p < 40 and pl_b >= 45 and pl_t <= 20) or (pl_p <= 20 and pl_b >= 45 and pl_t < 40): return "SS"
    if pl_p <= 20 and 40 <= pl_t < 52: return "TS"
    if pl_p <= 35 and pl_t >= 52: return "TF"
    return "U"

def stereo_xy(trend, plunge):
    """Lower-hemisphere equal-area projection of axes given by trend and plunge (deg); radius 1 = horizontal."""
    r = np.sqrt(2)*np.sin(np.radians(90 - np.asarray(plunge, float))/2)
    return r*np.sin(np.radians(trend)), r*np.cos(np.radians(trend))

def stereonet(ax, title=None):
    t = np.linspace(0, 2*np.pi, 361)
    ax.plot(np.cos(t), np.sin(t), "k", lw=1.2); ax.plot(0, 0, "k+", ms=6)
    for pl in (30, 60):                                                           # plunge circles
        r = np.sqrt(2)*np.sin(np.radians(90 - pl)/2); ax.plot(r*np.cos(t), r*np.sin(t), ":", color="0.6", lw=0.6)
    for a in range(0, 180, 30):
        ax.plot([-np.sin(np.radians(a)), np.sin(np.radians(a))], [-np.cos(np.radians(a)), np.cos(np.radians(a))], ":", color="0.6", lw=0.6)
    ax.text(0, 1.04, "N", ha="center", va="bottom"); ax.set_xlim(-1.12, 1.12); ax.set_ylim(-1.12, 1.15)
    ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=9)

# axes and faulting type of every mechanism of the catalog
_ax = [pbt_axes(r.strike, r.dip, r.rake) for r in catalog.itertuples()]
catalog[["P_trend", "P_plunge"]] = [a[0] for a in _ax]
catalog[["T_trend", "T_plunge"]] = [a[2] for a in _ax]
catalog["regime"] = [wsm_regime(a[0][1], a[1][1], a[2][1]) for a in _ax]

sel = catalog                                                                 # the mechanisms of the stress inversion
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.6), gridspec_kw=dict(width_ratios=[1, 1.15]))
stereonet(a1, f"P axes (red circles) and T axes (blue crosses)\n{len(sel)} mechanisms, lower hemisphere")
a1.scatter(*stereo_xy(sel.P_trend, sel.P_plunge), s=28, facecolors="none", edgecolors="r", linewidths=1.2, label="P")
a1.scatter(*stereo_xy(sel.T_trend, sel.T_plunge), s=34, marker="+", c="b", linewidths=1.2, label="T")
a1.legend(loc="lower right", fontsize=8)
order = list(REGIME_NAME)
counts = sel.regime.value_counts().reindex(order, fill_value=0)
a2.barh(range(len(order))[::-1], counts.values, color=["#4c72b0", "#64a0c8", "#55a868", "#e1a03c", "#c44e52", "0.6"])
a2.set_yticks(range(len(order))[::-1]); a2.set_yticklabels([f"{k}: {REGIME_NAME[k]}" for k in order], fontsize=8)
for y, c in zip(range(len(order))[::-1], counts.values): a2.text(c + 0.3, y, str(c), va="center", fontsize=8)
a2.set_xlabel("number of mechanisms"); a2.set_title(f"faulting type (World Stress Map classes)\n{len(sel)} mechanisms", fontsize=9)
plt.tight_layout(); plt.show()

#### Exercise 6 — Predict before you invert

Write your answers down; you will check them in 3.5.

- Which faulting type dominates the catalog? Is there only one type?
- Where do the P axes cluster (azimuth, plunge)? And the T axes? Where would you put σ₁ and σ₃?
- Which axis do you expect to be (close to) vertical: σ₁, σ₂ or σ₃? What tectonic regime is that (Anderson)?
- The Main Marmara Fault is the western continuation of the right-lateral North Anatolian Fault and runs roughly E–W here. Which orientation of σ₁ would make right-lateral slip on an E–W vertical fault easy?

### 3.2 What does a stress inversion do?

**The question.** Find the stress tensor that best explains the slip of *all* the earthquakes of the catalog.

**The assumptions:**

1. The stress is **uniform** in the volume and the time window of the earthquakes you use.
2. Every earthquake slips **in the direction of the shear stress** that this stress produces on its fault plane (the Wallace–Bott hypothesis).
3. The faults have **different orientations** (if all the mechanisms are the same, the stress cannot be resolved).
4. The earthquakes do not change the stress (they are small and independent).

**What can be determined: 4 numbers, not 6.** A stress tensor has 6 independent components. But the *direction* of the shear stress on a plane does not change if you add the same pressure in all directions, or multiply all the stresses by the same factor. So focal mechanisms only tell us:

- the **orientation** of the three principal stresses (3 angles): **σ₁** (most compressive), **σ₂** (intermediate), **σ₃** (least compressive), each given as azimuth and plunge;
- the **shape ratio** $R = \dfrac{\sigma_1-\sigma_2}{\sigma_1-\sigma_3}$, between 0 and 1: where σ₂ lies between σ₁ and σ₃. $R \approx 0$: σ₂ ≈ σ₁ (σ₁ and σ₂ can swap, e.g. normal ↔ strike-slip). $R \approx 1$: σ₂ ≈ σ₃ (σ₂ and σ₃ can swap, e.g. strike-slip ↔ thrust). *(Careful: other programs use $\phi = (\sigma_2-\sigma_3)/(\sigma_1-\sigma_3) = 1-R$.)*

The absolute values of the stresses (in MPa) **cannot** be obtained from focal mechanisms.

**The problem of the two nodal planes.** Each mechanism has two nodal planes and only one is the fault. Each plane would slip in a different direction in a given stress field, so the inversion must know which one slipped. **StressInverse** decides it with a physical criterion: in a trial stress field, the fault is the nodal plane that is **closer to failure** — the one with the larger **instability** *I* (Mohr–Coulomb criterion with a friction coefficient μ; *I* = 1 for a plane optimally oriented for failure, smaller for planes that are badly oriented).

**The algorithm** (Vavryčuk, 2014, based on the linear inversion of Michael, 1984, and the instability criterion of Lund & Slunga, 1999):

```
INPUT                     STRESSINVERSE                                          OUTPUT
N focal mechanisms   -->  1. first guess of the stress: linear inversion     -->  sigma1, sigma2, sigma3 (azimuth, plunge)
(strike, dip, rake)          (Michael, 1984) with randomly chosen nodal planes     shape ratio R
                          2. for every mechanism: instability I of both           friction mu
control parameters           nodal planes; the fault = the more unstable one       fault plane of every mechanism
(friction range,          3. invert again, using only these fault planes          the two "principal" (optimal) faults
 assumed error of the     4. repeat 2-3 (N_iterations times): the stress and     uncertainty of the sigmas and of R
 mechanisms, number of       the choice of the faults converge together
 noisy runs ...)          5. repeat 1-4 for mu = 0.40, 0.45 ... 0.95 and keep
                             the mu that makes the faults most unstable
                          6. add random errors to the mechanisms and repeat
                             everything (N_noise_realizations times) -> uncertainty
```

Michael's linear inversion (steps 1 and 3) is a least-squares fit of the 5 unknowns of the stress tensor (6 components minus the isotropic part), assuming that the shear stress has a similar size on all the faults; the result is scaled, so only the 4 numbers above are meaningful.

### 3.3 The inputs

**1. The focal mechanisms.** A plain text file with **three columns: strike, dip, rake** (degrees), one line per earthquake. Either nodal plane can be given: the program computes the other one, and the order of the lines does not matter. Here the file is `StressInverse/Data/Marmara_fms.dat` (inside `focal_stress_data/`): the 71 mechanisms of `mechanisms_catalog_2019.csv`, in the same order.

Nothing else is used: no location, no magnitude, no quality grade; every mechanism has the same weight. **Which mechanisms go into the file is your decision** (quality, area, depth, time window…), and it is often the most important choice of the whole analysis.

**2. The control parameters**, in `StressInverse/Programs_PYTHON/Input_parameters.py`:

| Parameter | Value here | Meaning | Effect on the result |
|---|---|---|---|
| `input_file` | `Data/Marmara_fms.dat` (relative to the `StressInverse/` folder) | file with the mechanisms | — |
| `output_file`, `principal_mechanisms_file` | `Output/Marmara_…` | where the results are written | — |
| `mean_deviation` | 5° | assumed error of the mechanisms: standard deviation of the random errors added in step 6 | **sets the size of the uncertainty** of σ₁, σ₂, σ₃ and R. Should match the real uncertainty of your mechanisms |
| `N_noise_realizations` | 100 | number of noisy inversions (step 6) | more = smoother uncertainty estimate (but slower) |
| `friction_min`, `friction_max`, `friction_step` | 0.40, 1.00, 0.05 | friction values tested (step 5) | usually left as is |
| `N_iterations` | 6 | iterations of steps 2–3 | usually left as is |
| `N_realizations` | 10 | random choices of nodal planes for the first guess (step 1) | usually left as is |
| `shape_ratio_min/max/step` | 0, 1, 0.025 | bins of the shape-ratio histogram | plot only |

The cell below shows the first lines of the input file and the values read from `Input_parameters.py`.

In [ ]:
# The StressInverse package: make its modules importable from the notebook
import importlib
SI_PROGRAMS = (STRESSINVERSE_DIR/"Programs_PYTHON").resolve()
if str(SI_PROGRAMS) not in sys.path:
    sys.path.insert(0, str(SI_PROGRAMS))
for _m in ("Input_parameters", "read_mechanism", "stress_inversion", "slip_deviation", "principal_mechanisms", "noisy_mechanisms",
           "statistics_stress_inversion", "plot_stress", "plot_mohr", "plot_stress_axes", "azimuth_plunge"):
    _mod = sys.modules.get(_m)                            # a module of the same name imported from another folder?
    if _mod is not None and Path(getattr(_mod, "__file__", "") or ".").resolve().parent != SI_PROGRAMS:
        del sys.modules[_m]                               # then use the one of this practical
import Input_parameters                                   # StressInverse's parameter file (its paths start from its own folder)
ip = importlib.reload(Input_parameters)                   # read the current values of the file

# INPUT 1: the mechanisms (file named in Input_parameters.py)
SI_INPUT = Path(ip.input_file)                           # absolute: built by Input_parameters.py from its own folder
SI_MECH = catalog                                         # the same mechanisms, as a table (same order as the file)
print(f"Input file {SI_INPUT}: {len(np.loadtxt(SI_INPUT, ndmin=2))} mechanisms. First lines (strike, dip, rake):")
print("".join(SI_INPUT.read_text().splitlines(keepends=True)[:5]))
# INPUT 2: the control parameters
for k in ("input_file", "output_file", "principal_mechanisms_file", "mean_deviation", "N_noise_realizations",
          "friction_min", "friction_max", "friction_step", "N_iterations", "N_realizations"):
    print(f"   {k:27s} = {getattr(ip, k)}")

### 3.4 Run StressInverse

`run_stress_inversion(mechanisms, name)` calls the functions of the StressInverse package **in the same order as its main script** `StressInverse.py` (steps 1–6 above), writes the same output files and figures, and returns the results so that we can print and compare them:

- `mechanisms`: the input file (as in `Input_parameters.py`), **or** a table with columns `strike, dip, rake` (e.g. a selection of the catalog; it is written to `StressInverse/Data/<name>_fms.dat` first);
- `name`: label of the run, used in the names of the output files (`StressInverse/Output/<name>_…`) and figures (`StressInverse/Figures/<name>_…`);
- optional: `mean_deviation`, `n_noise`, `friction=(min, max, step)`, `n_iterations`, `n_realizations`. What you do not give is read from `Input_parameters.py`. `seed` fixes the random numbers, so that everybody gets the same result.

> Outside the notebook, the package is run with `python StressInverse/Programs_PYTHON/StressInverse.py` (it reads `Input_parameters.py`, whose paths start from the `StressInverse/` folder, so it works from any folder). We do not `%run` that script inside the notebook: it switches matplotlib to a non-interactive backend (figures stop appearing in the notebook); with some IPython versions it also erases all the notebook's variables, with others it does not see later changes of `Input_parameters.py` until the kernel is restarted.

In [ ]:
# --- helper code for the stress inversion: just run this cell ---
import read_mechanism as si_read, stress_inversion as si_inv, slip_deviation as si_sd, principal_mechanisms as si_pm
import noisy_mechanisms as si_noise, statistics_stress_inversion as si_stat
import plot_stress as si_plot_stress, plot_mohr as si_plot_mohr, plot_stress_axes as si_plot_axes
import logging
from scipy.io import savemat
_ComplexWarning = getattr(np, "exceptions", np).ComplexWarning

def _axis_angle(a, b):
    """Angle (deg) between two axes (the sign of the vectors does not matter)."""
    return float(np.degrees(np.arccos(np.clip(abs(np.dot(a, b)), 0, 1))))

def shmax_wsm(regime, axes):
    """Azimuth of the maximum horizontal compression (World Stress Map rules), from the regime and the
    (trend, plunge) of sigma1, sigma2, sigma3."""
    (t1, p1), (t2, p2), (t3, p3) = axes
    if regime == "NF": return t2 % 180
    if regime == "NS": return (t3 + 90) % 180
    if regime == "SS": return ((t3 + 90) if (p1 < 40 and p2 >= 45 and p3 <= 20) else t1) % 180
    if regime in ("TS", "TF"): return t1 % 180
    return np.nan

def run_stress_inversion(mechanisms, name, mean_deviation=None, n_noise=None, friction=None,
                         n_iterations=None, n_realizations=None, seed=1, verbose=True):
    """Run StressInverse (Vavrycuk, 2014) step by step, like StressInverse.py.
    INPUT   mechanisms: input file (columns strike dip rake) or a table with columns strike, dip, rake
            name: label of the run (output file names); the other parameters default to Input_parameters.py
    OUTPUT  dictionary with the principal stress axes, shape ratio, friction, fault planes, uncertainties...
            + files in StressInverse/Output/<name>_* and figures in StressInverse/Figures/<name>_*"""
    ip = importlib.reload(Input_parameters)                                    # current values of Input_parameters.py
    mean_deviation = ip.mean_deviation if mean_deviation is None else mean_deviation
    n_noise = ip.N_noise_realizations if n_noise is None else n_noise
    n_iterations = ip.N_iterations if n_iterations is None else n_iterations
    n_realizations = ip.N_realizations if n_realizations is None else n_realizations
    fmin, fmax, fstep = (ip.friction_min, ip.friction_max, ip.friction_step) if friction is None else friction
    out_dir, fig_dir = STRESSINVERSE_DIR/"Output", STRESSINVERSE_DIR/"Figures"
    out_dir.mkdir(exist_ok=True); fig_dir.mkdir(exist_ok=True)
    np.random.seed(seed)                                                        # StressInverse uses numpy's random numbers

    # ---- INPUT: a file with three columns, strike dip rake (one line per mechanism)
    if isinstance(mechanisms, (str, Path)):
        input_file = Path(mechanisms)
    else:
        input_file = STRESSINVERSE_DIR/"Data"/f"{name}_fms.dat"
        np.savetxt(input_file, np.asarray(mechanisms[["strike", "dip", "rake"]], float), fmt="%10.2f")
    n_mech = len(np.loadtxt(input_file, comments=["#", "%"], ndmin=2))
    if n_mech < 5:
        raise ValueError(f"only {n_mech} mechanisms in {input_file}: a stress inversion needs many more")

    mpl_log = logging.getLogger("matplotlib"); mpl_level = mpl_log.level
    mpl_log.setLevel(logging.ERROR)                                             # silence aspect-ratio notes of the package plots
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", _ComplexWarning)
        # 1. read the mechanisms; the second nodal plane of each one is computed
        s1, d1, r1, s2, d2, r2 = si_read.read_mechanisms(str(input_file))
        # 2.-5. iterative joint inversion for the stress tensor, the fault plane of each mechanism and the friction
        tau, R, strike, dip, rake, instability, mu = si_inv.stress_inversion(
            s1, d1, r1, s2, d2, r2, fmin, fmax, fstep, n_iterations, n_realizations)
        tau, R, mu = np.real(tau), float(np.real(R)), float(np.real(np.ravel(mu)[0]))
        strike, dip, rake, instability = (np.real(x) for x in (strike, dip, rake, instability))
        # principal stresses = eigenvectors of the stress tensor (compression < 0: smallest eigenvalue = sigma1)
        _, vec = np.linalg.eigh(tau)
        axes = [trend_plunge(vec[:, k]) for k in range(3)]
        # misfit: angle between the observed slip and the shear stress on each selected fault plane
        slip_dev, _ = si_sd.slip_deviation(tau, strike, dip, rake)
        # the two optimally oriented ("principal") faults for this stress and friction
        p_strike, p_dip, p_rake = (np.real(x) for x in si_pm.principal_mechanisms(vec[:, 0], vec[:, 2], mu))
        # 6. uncertainty: random errors (standard deviation = mean_deviation) added to the mechanisms, inversion repeated
        noisy_vec, noisy_R = np.zeros((n_noise, 3, 3)), np.zeros(n_noise)
        for i in range(n_noise):
            noisy = si_noise.noisy_mechanisms(mean_deviation, s1, d1, r1)
            v1, v2, v3, Rn = si_stat.statistics_stress_inversion(*noisy[:6], mu, n_iterations, n_realizations)
            noisy_vec[i], noisy_R[i] = np.real([v1, v2, v3]), np.real(Rn)
        err = np.array([[_axis_angle(noisy_vec[i, k], vec[:, k]) for k in range(3)] for i in range(n_noise)])

        # ---- OUTPUT files, as written by StressInverse.py
        sig = {f"sigma_{k + 1}": {"azimuth": axes[k][0], "plunge": axes[k][1]} for k in range(3)}
        savemat(out_dir/f"{name}_output.mat", {**sig, "shape_ratio": R, "friction": mu,
                "mechanisms": {"strike": strike, "dip": dip, "rake": rake}, "mechanisms_data": np.column_stack([strike, dip, rake]),
                "principal_mechanisms": {"strike": p_strike, "dip": p_dip, "rake": p_rake}})
        np.savetxt(out_dir/f"{name}_output.dat", np.column_stack([strike, dip, rake]), fmt="%1.4e")
        np.savetxt(out_dir/f"{name}_principal_mechanisms.dat", np.column_stack([p_strike, p_dip, p_rake]), fmt="%1.4e")
        # ---- the four figures of StressInverse
        figs = {k: fig_dir/f"{name}_{k}.png" for k in ("P_T_axes", "stress_directions", "shape_ratio", "Mohr_circles")}
        si_plot_stress.plot_stress(tau, strike, dip, rake, str(fig_dir/f"{name}_P_T_axes"))
        si_plot_axes.plot_stress_axes(noisy_vec[:, 0].T, noisy_vec[:, 1].T, noisy_vec[:, 2].T, str(fig_dir/f"{name}_stress_directions"))
        si_plot_mohr.plot_mohr(tau, strike, dip, rake, p_strike, p_dip, p_rake, str(fig_dir/f"{name}_Mohr_circles"))
        fig, axh = plt.subplots()
        axh.hist(noisy_R, bins=ip.shape_ratio_axis, color="#0504aa", alpha=0.7, rwidth=0.85)
        axh.set_title("Shape ratio", fontsize=14); axh.grid(True); fig.savefig(figs["shape_ratio"]); plt.close(fig)
    mpl_log.setLevel(mpl_level)

    regime = wsm_regime(axes[0][1], axes[1][1], axes[2][1])
    res = dict(name=name, n=len(s1), input_file=input_file, mean_deviation=mean_deviation, n_noise=n_noise, seed=seed,
               planes=(s1, d1, r1, s2, d2, r2), n_iterations=n_iterations, n_realizations=n_realizations, friction_range=(fmin, fmax, fstep),
               tau=tau, vectors=vec, axes=axes, R=R, friction=mu, regime=regime, shmax=shmax_wsm(regime, axes),
               faults=pd.DataFrame(dict(strike=strike, dip=dip, rake=rake, instability=instability, slip_deviation=slip_dev)),
               principal=pd.DataFrame(dict(strike=p_strike, dip=p_dip, rake=p_rake), index=["principal fault 1", "principal fault 2"]),
               noisy_vectors=noisy_vec, noisy_R=noisy_R, err95=np.percentile(err, 95, axis=0), R90=np.percentile(noisy_R, [5, 95]),
               figures=figs, files=[out_dir/f"{name}_output.mat", out_dir/f"{name}_output.dat", out_dir/f"{name}_principal_mechanisms.dat"])
    if verbose: stress_summary(res)
    return res

def stress_summary(res):
    lab = ["sigma1 (most compressive) ", "sigma2 (intermediate)     ", "sigma3 (least compressive)"]
    print(f"StressInverse run '{res['name']}': {res['n']} mechanisms, assumed error {res['mean_deviation']}°, {res['n_noise']} noisy runs")
    for k in range(3):
        print(f"   {lab[k]}: azimuth {res['axes'][k][0]:5.0f}°, plunge {res['axes'][k][1]:3.0f}°"
              f"   (95% of the noisy runs within {res['err95'][k]:.0f}°)")
    print(f"   shape ratio R = {res['R']:.2f}   (90% of the noisy runs between {res['R90'][0]:.2f} and {res['R90'][1]:.2f})")
    print(f"   friction mu = {res['friction']:.2f}")
    print(f"   regime: {res['regime']} = {REGIME_NAME[res['regime']]}   (SHmax ≈ {res['shmax']:.0f}°, World Stress Map rules)")
    print(f"   misfit: median slip deviation {res['faults'].slip_deviation.median():.0f}° "
          f"(angle between the observed slip and the shear stress predicted on each fault)")
    print("   output files: " + ", ".join(str(f) for f in res["files"]))
    print("   figures:      " + ", ".join(str(f) for f in res["figures"].values()))

def show_stress_figures(res):
    """The four figures written by StressInverse for one run."""
    titles = {"P_T_axes": "P (red o) and T (blue +) axes and the best stress axes (green: o σ1, x σ2, + σ3)",
              "stress_directions": f"σ1 (red), σ2 (green), σ3 (blue) of the {res['n_noise']} noisy runs = uncertainty",
              "shape_ratio": "shape ratio R of the noisy runs", "Mohr_circles": "Mohr circles: one + per fault plane"}
    fig, axs = plt.subplots(2, 2, figsize=(12, 9.5))
    for ax, (k, path) in zip(axs.flat, res["figures"].items()):
        ax.imshow(plt.imread(path)); ax.axis("off"); ax.set_title(titles[k], fontsize=9)
    fig.suptitle(f"StressInverse — run '{res['name']}' ({res['n']} mechanisms)", fontweight="bold")
    plt.tight_layout(); plt.show()

def compare_stress(runs):
    """Table and stereonets of several runs of run_stress_inversion (differences relative to the first run)."""
    ref = runs[0]; rows = []
    for r in runs:
        ax = r["axes"]
        rows.append({"run": r["name"], "n": r["n"], "error_deg": r["mean_deviation"],
                     "sigma1 az/pl": f"{ax[0][0]:.0f}/{ax[0][1]:.0f}", "sigma2 az/pl": f"{ax[1][0]:.0f}/{ax[1][1]:.0f}",
                     "sigma3 az/pl": f"{ax[2][0]:.0f}/{ax[2][1]:.0f}", "R": f"{r['R']:.2f} ({r['R90'][0]:.2f}–{r['R90'][1]:.2f})",
                     "mu": round(r["friction"], 2), "regime": r["regime"], "SHmax": round(r["shmax"]) if np.isfinite(r["shmax"]) else None,
                     "sigma1 95% (deg)": round(r["err95"][0]), "sigma3 95% (deg)": round(r["err95"][2]),
                     "d_sigma1 vs 1st": round(_axis_angle(r["vectors"][:, 0], ref["vectors"][:, 0])),
                     "d_sigma3 vs 1st": round(_axis_angle(r["vectors"][:, 2], ref["vectors"][:, 2])),
                     "misfit (deg)": round(r["faults"].slip_deviation.median())})
    display(pd.DataFrame(rows).set_index("run"))
    fig, axs = plt.subplots(1, len(runs), figsize=(3.3*len(runs), 3.8), squeeze=False)
    for a, r in zip(axs[0], runs):
        stereonet(a, f"{r['name']}\n{r['n']} mech., {r['regime']}, R = {r['R']:.2f}")
        for k, c in enumerate(("r", "g", "b")):
            tp = np.array([trend_plunge(v) for v in r["noisy_vectors"][:, k]])
            a.scatter(*stereo_xy(tp[:, 0], tp[:, 1]), s=6, c=c, alpha=0.35, lw=0)
            a.scatter(*stereo_xy(*r["axes"][k]), s=90, c=c, ec="k", lw=1.2, zorder=5, label=f"σ{k + 1}")
    axs[0][0].legend(loc="lower left", fontsize=7, ncol=3, bbox_to_anchor=(0, -0.12), frameon=False)
    plt.tight_layout(); plt.show()

In [ ]:
SI = run_stress_inversion(SI_INPUT, name="Marmara2019")    # the 71 mechanisms, parameters of Input_parameters.py

### 3.5 The outputs

| Output | Where | Meaning |
|---|---|---|
| **σ₁, σ₂, σ₃**: azimuth and plunge | printed above; `Output/<name>_output.mat` | orientation of the principal stresses. σ₁ is the most compressive |
| **shape ratio R** | printed; `.mat`; `Figures/<name>_shape_ratio.png` | $R=(\sigma_1-\sigma_2)/(\sigma_1-\sigma_3)$: relative size of σ₂ (0 = σ₂ like σ₁, 1 = σ₂ like σ₃) |
| **friction μ** | printed; `.mat` | the friction value (0.40–0.95) that makes the selected faults most unstable. Poorly constrained: do not over-interpret it |
| **fault plane of every mechanism** | `Output/<name>_output.dat` (strike, dip, rake, same order as the input) | which of the two nodal planes the inversion chose as the fault |
| **principal mechanisms** | `Output/<name>_principal_mechanisms.dat` (2 lines: strike, dip, rake) | the two conjugate faults that are *optimally* oriented for slip in this stress field |
| **uncertainty** | printed (95% of the noisy runs); `Figures/<name>_stress_directions.png`, `_shape_ratio.png` | spread of σ₁, σ₂, σ₃ and R when random errors of size `mean_deviation` are added to the mechanisms |
| regime, SHmax, misfit | printed (computed in this notebook, not by StressInverse) | Andersonian regime from the plunges (World Stress Map classes), azimuth of the maximum horizontal compression, and the median angle between the observed slip and the shear stress predicted on each fault |

**How to read the four figures** (all stereonets are lower hemisphere, equal area, north up):

- **P_T_axes**: P (red circles) and T (blue crosses) axes of all the mechanisms, with the best σ₁ (green circle), σ₂ (green ×) and σ₃ (green +). σ₁ should sit inside the cloud of P axes and σ₃ inside the cloud of T axes — but not necessarily at their center.
- **stress_directions**: σ₁ (red), σ₂ (green) and σ₃ (blue) of each noisy inversion. Small, separate clouds = well-resolved axes. Two clouds that merge (e.g. σ₂ and σ₃) = these two axes can be exchanged (look at R!).
- **shape_ratio**: histogram of R in the noisy inversions.
- **Mohr_circles**: the three Mohr circles of the stress (normalized; σ₁ at the right end, σ₃ at the left end), and one + for every selected fault plane at its normal stress (horizontal axis) and shear stress (vertical axis). The best-oriented faults plot near the top of the outer circle, a bit towards σ₃ (high shear stress, moderate normal stress); the upper and lower halves are mirror images (faults closer to the first or to the second principal fault). Faults far inside the circles are badly oriented: wrong nodal plane, poor mechanism, or the stress is not uniform.

In [ ]:
show_stress_figures(SI)
print(f"Fault plane chosen for the first 5 mechanisms (Output/{SI['name']}_output.dat), with its instability and misfit:")
display(SI["faults"].head().round(1))
print(f"The two principal (optimally oriented) faults (Output/{SI['name']}_principal_mechanisms.dat):")
display(SI["principal"].round(1))

#### Exercise 7 — Interpretation

Compare with your predictions of 3.1.

- Which principal stress is closest to vertical? What is the faulting regime (Anderson: vertical σ₁ → normal, vertical σ₂ → strike-slip, vertical σ₃ → thrust)? Is it pure, or between two regimes?
- Compare σ₁ and σ₃ with the clouds of P and T axes. Are they at the center of the clouds? Why not necessarily?
- Is σ₁ oriented as you expected for right-lateral slip on the roughly E–W Main Marmara Fault? Compare the principal mechanisms with the strike of the fault.
- What does the value of R tell you? Which two axes could be exchanged?
- How large is the uncertainty? It was computed with an assumed error of the mechanisms of 5°. Look at the column `fault_plane_uncertainty` of the catalog (median ≈ 25°): is 5° realistic? (You will test it in 3.6.)
- The median misfit is the typical angle between the observed slip and the predicted one. Is one uniform stress field a good description of all these earthquakes?

### 3.6 Is the result reliable?

A stress inversion **always** returns a stress tensor, even for mechanisms that have nothing in common. Before you interpret it, check it from several sides:

| # | Check | Question | How | Good sign |
|---|---|---|---|---|
| 1 | **Precision** | How far do the errors of the mechanisms move the result? | the noisy runs of StressInverse (step 6), with an error that matches the real one: `mean_deviation=25`, close to the median nodal-plane uncertainty of the catalog | small, separate clouds of σ₁, σ₂, σ₃ |
| 2 | **Fit** | Can one uniform stress explain all these earthquakes? | the misfit of every mechanism: angle between its observed slip and the shear stress predicted on its fault | most misfits not larger than the errors of the mechanisms; no large group of badly fitted events |
| 3 | **Sampling** | Would other earthquakes of the same kind give the same answer? | bootstrap: draw the mechanisms at random *with replacement* (same number; some appear twice, some not at all) and invert again, 200 times | the same small clouds |
| 4 | **Method** | Can the method find a stress that we **know**, with these fault orientations and these errors? | synthetic test: invent the slip that a known stress would produce on the same faults, add errors, invert | the known stress comes back, within a few degrees |
| 5 | **Choices** | Does the answer depend on my choices (region, quality, time window…)? | section 3.7 | changes smaller than the uncertainty |

Checks 1–4 test the **data and the method**, not the assumptions: the result can pass them all and still be wrong if the stress is not uniform, or if faults do not slip in the direction of the shear stress. So the last check is always **physical**: does the result agree with the P and T axes and with the tectonics (right-lateral slip on the E–W Main Marmara Fault)? (Martínez-Garzón et al., 2016, discuss how the number, quality and diversity of the mechanisms affect stress inversions.)

In [ ]:
# --- helper code for the reliability checks: just run this cell ---
def plot_misfit(res):
    """Check 2 (fit): histogram of the angle between the observed slip and the shear stress predicted on each fault."""
    sd = res["faults"].slip_deviation
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.hist(sd, bins=np.arange(0, 95, 5), color="C0", ec="k")
    ax.axvline(sd.median(), color="r", ls="--", label=f"median {sd.median():.0f}°")
    ax.set_xlabel("misfit = angle between the observed slip and the predicted shear stress (°)")
    ax.set_ylabel("number of mechanisms"); ax.legend(fontsize=8)
    ax.set_title(f"run '{res['name']}': misfit of the {len(sd)} mechanisms", fontsize=9)
    plt.tight_layout(); plt.show()
    print(f"misfit: median {sd.median():.0f}°, 90% of the mechanisms below {sd.quantile(0.9):.0f}°, "
          f"{(sd > 30).sum()} of {len(sd)} above 30°")

def bootstrap_stress(res, n_boot=200, seed=0):
    """Check 3 (sampling): draw the mechanisms at random WITH replacement (same number, some twice, some not at all)
    and invert again, n_boot times, with the friction of the run. Spread = how much the result depends on which
    mechanisms we happen to have."""
    s1, d1, r1, s2, d2, r2 = (np.asarray(x, float) for x in res["planes"])
    rng = np.random.default_rng(seed); np.random.seed(seed)
    vecs, Rs = np.zeros((n_boot, 3, 3)), np.zeros(n_boot)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                                           # harmless numerical warnings of the package
        for b in range(n_boot):
            k = rng.integers(0, len(s1), len(s1))
            v1, v2, v3, R = si_stat.statistics_stress_inversion(s1[k], d1[k], r1[k], s2[k], d2[k], r2[k], res["friction"],
                                                               res["n_iterations"], res["n_realizations"])
            vecs[b], Rs[b] = np.real([v1, v2, v3]), np.real(R)
    err = np.array([[_axis_angle(vecs[b, j], res["vectors"][:, j]) for j in range(3)] for b in range(n_boot)])
    return dict(name=f"bootstrap of '{res['name']}'", vectors=vecs, R=Rs, err95=np.percentile(err, 95, axis=0),
                R90=np.percentile(Rs, [5, 95]))

def andersonian_stress(regime, shmax, R=0.5):
    """A stress tensor to test the method: regime 'NF', 'SS' or 'TF', maximum horizontal compression towards
    shmax (deg), shape ratio R. Compression negative, as in StressInverse."""
    a = np.radians(shmax)
    h1, h2, v = np.array([np.cos(a), np.sin(a), 0.0]), np.array([-np.sin(a), np.cos(a), 0.0]), np.array([0.0, 0.0, 1.0])
    s1, s2, s3 = {"NF": (v, h1, h2), "SS": (h1, v, h2), "TF": (h1, h2, v)}[regime]
    V = np.column_stack([s1, s2, s3])
    return V @ np.diag([-1.0, -1.0 + 2*R, 1.0]) @ V.T

def synthetic_stress_test(res, truth=None, mean_deviation=20, n_tests=50, seed=0):
    """Check 4 (method): a test with a KNOWN answer.
    1. take the fault planes of the run and compute the slip that the true stress would produce on them
       (slip parallel to the shear stress: the assumption of the method);
    2. add random errors of size mean_deviation (deg) to these synthetic mechanisms;
    3. invert them exactly like the real data (both nodal planes, friction search, instability criterion);
    4. compare with the true stress. Repeated n_tests times with different random errors.
    truth: a stress tensor (e.g. andersonian_stress("TF", 0)); default: the stress found by the run."""
    tau_true = res["tau"] if truth is None else np.asarray(truth, float)
    w, V = np.linalg.eigh(tau_true); R_true = (w[0] - w[1])/(w[0] - w[2])
    f = res["faults"]; s, d = np.radians(f.strike.to_numpy()), np.radians(f.dip.to_numpy())
    n = np.column_stack([-np.sin(d)*np.sin(s), np.sin(d)*np.cos(s), -np.cos(d)])         # fault normals (upwards)
    t = n @ tau_true.T; sh = t - np.sum(t*n, 1)[:, None]*n                                # traction -> shear traction
    u = sh/np.linalg.norm(sh, axis=1)[:, None]                                            # slip of the synthetic events
    syn = np.array([si_noise.strike_dip_rake(n[i], u[i])[:3] for i in range(len(n))], float)
    np.random.seed(seed)
    fmin, fmax, fstep = res["friction_range"]
    errs, Rs, ok_plane = [], [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                                           # harmless numerical warnings of the package
        for k in range(n_tests):
            sn, dn, rn, sa, da, ra, _, _ = si_noise.noisy_mechanisms(mean_deviation, syn[:, 0], syn[:, 1], syn[:, 2])
            tau, R, st, dp, rk, _, _ = si_inv.stress_inversion(sn, dn, rn, sa, da, ra, fmin, fmax, fstep,
                                                              res["n_iterations"], res["n_realizations"])
            _, vec = np.linalg.eigh(np.real(tau))
            errs.append([_axis_angle(vec[:, j], V[:, j]) for j in range(3)]); Rs.append(float(np.real(R)))
            same = (np.abs((np.real(st) - sn + 180) % 360 - 180) < 1) & (np.abs(np.real(dp) - dn) < 1)
            ok_plane.append(same.mean())
    errs = np.array(errs)
    print(f"Synthetic test: {len(n)} mechanisms on the fault planes of '{res['name']}', true stress = "
          f"{'the result of the run' if truth is None else 'the tensor you gave'}, random errors {mean_deviation}°, {n_tests} tests")
    for j, lab in enumerate(("sigma1", "sigma2", "sigma3")):
        tr, pl = trend_plunge(V[:, j])
        print(f"   true {lab} {tr:5.0f}/{pl:2.0f}: recovered within {np.median(errs[:, j]):4.1f}° (median), "
              f"{np.percentile(errs[:, j], 95):4.1f}° (95% of the tests)")
    print(f"   true R = {R_true:.2f}: recovered R between {np.percentile(Rs, 5):.2f} and {np.percentile(Rs, 95):.2f} (90% of the tests)")
    print(f"   fault plane (out of the two nodal planes) chosen correctly: {100*np.mean(ok_plane):.0f}% of the mechanisms")
    return dict(err=errs, R=np.array(Rs), R_true=R_true, correct_plane=float(np.mean(ok_plane)))

def plot_uncertainty_clouds(res, others, titles=None):
    """The best stress axes of a run with the clouds of other tests (noisy runs, bootstrap...), side by side."""
    fig, axs = plt.subplots(1, len(others), figsize=(3.7*len(others), 4.5), squeeze=False)
    for a, o, t in zip(axs[0], others, titles or [o["name"] for o in others]):
        stereonet(a, t)
        for k, c in enumerate(("r", "g", "b")):
            tp = np.array([trend_plunge(v) for v in o["vectors"][:, k]])
            a.scatter(*stereo_xy(tp[:, 0], tp[:, 1]), s=6, c=c, alpha=0.35, lw=0)
            a.scatter(*stereo_xy(*res["axes"][k]), s=90, c=c, ec="k", lw=1.2, zorder=5, label=f"σ{k + 1}")
        a.text(0, -1.25, f"95%: σ1 {o['err95'][0]:.0f}°, σ3 {o['err95'][2]:.0f}°;  R {o['R90'][0]:.2f}–{o['R90'][1]:.2f}",
               ha="center", fontsize=7.5)
    h, l = axs[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=8, frameon=False)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.86, bottom=0.12, wspace=0.08); plt.show()

def reliability_summary(res, res_err, boot, syn):
    """One line per check (95% ranges, R: 90%)."""
    sd = res["faults"].slip_deviation; e = syn["err"]
    rows = [(f"1. precision: noisy mechanisms, {res['mean_deviation']}° errors",
             f"σ1 {res['err95'][0]:.0f}°, σ3 {res['err95'][2]:.0f}°;  R {res['R90'][0]:.2f}–{res['R90'][1]:.2f}"),
            (f"1. precision: noisy mechanisms, {res_err['mean_deviation']}° errors",
             f"σ1 {res_err['err95'][0]:.0f}°, σ3 {res_err['err95'][2]:.0f}°;  R {res_err['R90'][0]:.2f}–{res_err['R90'][1]:.2f}"),
            ("2. fit: misfit of the mechanisms", f"median {sd.median():.0f}°;  {100*(sd > 30).mean():.0f}% of the mechanisms above 30°"),
            ("3. sampling: bootstrap", f"σ1 {boot['err95'][0]:.0f}°, σ3 {boot['err95'][2]:.0f}°;  R {boot['R90'][0]:.2f}–{boot['R90'][1]:.2f}"),
            ("4. method: synthetic test", f"σ1 {np.percentile(e[:, 0], 95):.0f}°, σ3 {np.percentile(e[:, 2], 95):.0f}°;  "
             f"R {np.percentile(syn['R'], 5):.2f}–{np.percentile(syn['R'], 95):.2f} (true {syn['R_true']:.2f});  "
             f"right fault plane {100*syn['correct_plane']:.0f}%")]
    display(pd.DataFrame(rows, columns=["check", f"run '{res['name']}': spread of σ1 / σ3 (95%) and of R (90%)"]).set_index("check"))

**1 and 3 — precision and sampling.** Each panel shows the best σ₁, σ₂, σ₃ of the run (large symbols) and the cloud of one test: noisy mechanisms with 5° errors (the run of 3.4), with 25° errors, and the bootstrap.

In [ ]:
SI25 = run_stress_inversion(SI_INPUT, name="Marmara2019_err25", mean_deviation=25, verbose=False)   # check 1, realistic errors
BOOT = bootstrap_stress(SI, n_boot=200)                                                               # check 3
plot_uncertainty_clouds(SI, [dict(vectors=SI["noisy_vectors"], err95=SI["err95"], R90=SI["R90"]),
                             dict(vectors=SI25["noisy_vectors"], err95=SI25["err95"], R90=SI25["R90"]), BOOT],
                        ["1. noisy mechanisms, 5° errors", "1. noisy mechanisms, 25° errors", "3. bootstrap (200 draws)"])

**2 — fit.** For every mechanism, the angle between its observed slip and the shear stress that the best stress predicts on its fault plane (StressInverse calls it *slip deviation*). Compare it with the uncertainty of the mechanisms (column `fault_plane_uncertainty` of the catalog).

In [ ]:
plot_misfit(SI)
print(f"for comparison, nodal-plane uncertainty of these mechanisms: median {SI_MECH.fault_plane_uncertainty.median():.0f}°")

**4 — method: a test with a known answer.** `synthetic_stress_test` keeps the fault planes chosen in the run, computes the slip that a stress *we choose* would produce on them (slip parallel to the shear stress, the assumption of the method), adds random errors of `mean_deviation` degrees, and inverts these synthetic mechanisms exactly like the real ones (both nodal planes given, friction search, instability criterion), 50 times. By default the "true" stress is the result of the run: can the method get it back from data like ours? It also counts how often the inversion picks the right nodal plane as the fault.

In [ ]:
SYN = synthetic_stress_test(SI, mean_deviation=25)
# your own known stress, e.g. thrust faulting with N-S compression:
# synthetic_stress_test(SI, truth=andersonian_stress("TF", shmax=0, R=0.5), mean_deviation=25)

#### Exercise 8 — Summary of the checks

95% of the tests within these angles of the best σ₁ and σ₃; 90% range of R.

- Which axis is resolved best? Which two axes are hard to tell apart, and how does that agree with the value of R?
- Is the uncertainty dominated by the errors of the mechanisms (check 1) or by which mechanisms we happen to have (check 3)?
- In the synthetic test, is R recovered without a bias? What does that mean for the interpretation of R?
- How often is the fault plane chosen correctly? Does a wrong choice for some mechanisms ruin the result?
- Write one sentence on the reliability of the result, as you would in a paper.

In [ ]:
reliability_summary(SI, SI25, BOOT, SYN)

### 3.7 How robust is the result? Change the inputs

The result also depends on the **inputs you choose**. Re-run the inversion with other inputs and compare. `compare_stress([...])` gives one line per run (`d_sigma1`, `d_sigma3`: angle to the axes of the first run) and one stereonet per run with the clouds of its noisy runs.

| Experiment | How | Question |
|---|---|---|
| Only the best mechanisms | `catalog[catalog.quality.isin(["A", "B"])]` | Do the quality-C mechanisms bias the result, or only make it noisier? |
| Realistic errors | `mean_deviation=25` (run `SI25` of 3.6) | How big is the uncertainty when the assumed error matches the real one? |
| Few mechanisms | `catalog.sample(15, random_state=1)` (try other numbers and `random_state`s) | How many mechanisms do you need for a stable result? |
| Space | `catalog[catalog.origin_lon < 28.24]` and `catalog[catalog.origin_lon >= 28.24]` (western and eastern part) | Is the stress the same along the rupture? |
| One faulting type only | `catalog[catalog.regime == "SS"]` | What happens when the fault orientations are not diverse? |
| Time, depth | `catalog[catalog.time < "2019-09-27"]` (until the day after the mainshock), `catalog[catalog.origin_depth_km > 6]` | Did the stress change during the sequence? Is it the same at depth? |

In [ ]:
RUNS_SI = [SI,
           run_stress_inversion(catalog[catalog.quality.isin(["A", "B"])], name="quality_AB", verbose=False),
           SI25,
           run_stress_inversion(catalog.sample(15, random_state=1), name="random15", verbose=False),
           run_stress_inversion(catalog[catalog.origin_lon < 28.24], name="west", verbose=False),
           run_stress_inversion(catalog[catalog.origin_lon >= 28.24], name="east", verbose=False)]
compare_stress(RUNS_SI)

### 3.8 Which earthquakes fit the stress field?

The inversion found the stress that fits the 2019 mechanisms *on average*. Now look at them one by one. For every mechanism, the misfit is the angle between its observed slip and the shear stress predicted on a nodal plane. The next cell computes it on **both** nodal planes and keeps the smaller one (the "best nodal plane": the one whose slip this stress explains better), and compares it with the misfit on the plane chosen by the inversion in 3.4 (the more unstable plane). It then maps the mechanisms colored by misfit, with the mapped faults and the orientation of the stress: the maximum horizontal compression S<sub>Hmax</sub> (black arrows pointing inwards), the minimum horizontal stress S<sub>hmin</sub> (blue arrows pointing outwards) and the 95% range of S<sub>Hmax</sub> from the bootstrap of 3.6 (gray fan). The stress symbol is drawn like the beachballs (north up, true azimuths) in an empty corner of the map, which covers the same area as the map at the start of Part 3. S<sub>Hmax</sub> is computed from the full stress tensor, as the horizontal direction with the largest compression (Lund & Townend, 2007); it can differ by a few degrees from the World Stress Map rule printed in 3.4.

#### Exercise 9

- Which faulting types fit the stress best, and which worst? Where are the badly fitted events?
- Do the quality-A mechanisms fit better than B and C? What does that suggest about the misfit of the others?
- For many mechanisms, the plane chosen by the inversion is not the best-fitting one. Why can that happen? (Recall how the inversion chooses the fault plane, 3.2.)
- How do S<sub>Hmax</sub> and S<sub>hmin</sub> relate to the strike of the mapped faults? Is this orientation favorable for right-lateral slip on them?

In [ ]:
# --- helper code for 3.8 (misfit of each mechanism, S_Hmax, map): run the cell; the analysis is at the end ---
def plane_misfits(res, strike, dip, rake):
    """Misfit (deg) between the slip of mechanisms and the shear stress of a run, on nodal plane 1 and on plane 2."""
    s, d, r = (np.radians(np.asarray(x, float)) for x in (strike, dip, rake))
    n = np.stack([-np.sin(d)*np.sin(s), np.sin(d)*np.cos(s), -np.cos(d)], -1)
    u = np.stack([np.cos(r)*np.cos(s) + np.cos(d)*np.sin(r)*np.sin(s), np.cos(r)*np.sin(s) - np.cos(d)*np.sin(r)*np.cos(s),
                  -np.sin(r)*np.sin(d)], -1)
    out = []
    for nn, uu in ((n, u), (u, n)):                       # plane 1: normal n, slip u;  plane 2: normal u, slip n
        sg = np.where(nn[:, 2] > 0, -1.0, 1.0)[:, None]; nn, uu = nn*sg, uu*sg          # upward normal, as in StressInverse
        t = nn @ res["tau"].T; sh = t - np.sum(t*nn, 1)[:, None]*nn                      # shear traction on the plane
        out.append(np.degrees(np.arccos(np.clip(np.sum(sh*uu, 1)/np.linalg.norm(sh, axis=1), -1, 1))))
    return out[0], out[1]

def shmax_from_tau(tau):
    """Azimuth (0-180°) of the maximum horizontal compression of a stress tensor (compression negative)."""
    w, v = np.linalg.eigh(np.asarray(tau)[:2, :2])                      # horizontal (north, east) part of the tensor
    return np.degrees(np.arctan2(v[1, 0], v[0, 0])) % 180

def shmax_range(res, boot):
    """S_Hmax of a run and the 95% range of the bootstrap (tensors rebuilt from their axes and R)."""
    sh0 = shmax_from_tau(res["tau"]); dev = []
    for V, R in zip(boot["vectors"], boot["R"]):
        tau_b = sum(s*np.outer(V[k], V[k]) for k, s in enumerate((-1.0, -1.0 + 2*R, 1.0)))
        dev.append((shmax_from_tau(tau_b) - sh0 + 90) % 180 - 90)
    return sh0, np.percentile(dev, [2.5, 97.5])

def fault_trend(x0, x1, y0, y1):
    """Mean strike (0-180°, weighted by length) of the mapped fault segments inside a map box."""
    k = np.cos(np.radians((y0 + y1)/2)); z = 0j
    for seg in FAULTS:
        seg = np.asarray(seg, float); mid = (seg[:-1] + seg[1:])/2
        inside = (mid[:, 0] >= x0) & (mid[:, 0] <= x1) & (mid[:, 1] >= y0) & (mid[:, 1] <= y1)
        dx, dy = np.diff(seg[:, 0])*k, np.diff(seg[:, 1])
        z += np.sum((np.hypot(dx, dy)*np.exp(2j*np.arctan2(dx, dy)))[inside])          # axial mean: doubled angles
    return np.degrees(np.angle(z))/2 % 180

def map_extent(mech):
    """The extent of the map at the start of Part 3: the mechanisms plus a margin."""
    lat_c = mech.origin_lat.mean(); half = max(0.07, (mech.origin_lat.max() - mech.origin_lat.min())/2 + 0.03)
    return mech.origin_lon.min() - 0.08, mech.origin_lon.max() + 0.08, lat_c - half, lat_c + half

def stress_map(res, boot, mech, title):
    """Mechanisms colored by their misfit (column misfit) with the stress of a run, mapped faults, and the stress
    orientation: S_Hmax (black arrows pointing in), S_hmin (blue arrows pointing out), 95% range of S_Hmax (grey fan).
    Same extent as the map at the start of Part 3."""
    pc = ccrs.PlateCarree(); fig = plt.figure(figsize=(11, 7.5)); ax = plt.axes(projection=pc)
    x0, x1, y0, y1 = map_extent(mech)
    ax.set_extent([x0, x1, y0, y1], crs=pc)
    ax.add_feature(cfeature.LAND, alpha=0.4); ax.add_feature(cfeature.OCEAN, alpha=0.3); ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    gl = ax.gridlines(draw_labels=True, alpha=0.3); gl.top_labels = gl.right_labels = False
    add_faults(ax, transform=pc, linewidths=1.0)
    cmap, norm = plt.cm.RdYlGn_r, plt.Normalize(0, 60)
    for _, e in mech.sort_values("magnitude", ascending=False).iterrows():
        ax.add_collection(beach([e.strike, e.dip, e.rake], xy=(e.origin_lon, e.origin_lat), width=15*1.5**(e.magnitude - 2),
                                linewidth=0.5, facecolor=cmap(norm(e.misfit)), zorder=3, axes=ax))
    star(ax, MAINSHOCK_2019["lat"], MAINSHOCK_2019["lon"], MAINSHOCK_2019["label"], transform=pc)
    # the stress symbol (drawn like the beachballs: true azimuths, north up) in the corner of the map farthest from the
    # earthquakes and the faults, the legend in the next one
    pts = [np.column_stack([mech.origin_lon, mech.origin_lat])] + [np.asarray(f, float) for f in FAULTS]
    pts = np.vstack(pts); pts = pts[(pts[:, 0] >= x0) & (pts[:, 0] <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)]
    fx, fy = (pts[:, 0] - x0)/(x1 - x0), (pts[:, 1] - y0)/(y1 - y0)                  # positions as fractions of the map
    ratio = (y1 - y0)/(x1 - x0)                                                       # height/width of the map
    corners = {"upper right": (0.88, 0.80), "upper left": (0.12, 0.80), "lower right": (0.88, 0.20), "lower left": (0.12, 0.20)}
    free = sorted(corners, key=lambda c: -np.min(np.hypot(fx - corners[c][0], (fy - corners[c][1])*ratio)))
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    sh0, (lo, hi) = shmax_range(res, boot)
    iax = inset_axes(ax, width=1.45, height=1.45, loc=free[0], borderpad=0.6)
    iax.set_xlim(-1.25, 1.25); iax.set_ylim(-1.25, 1.25); iax.set_aspect("equal"); iax.axis("off")
    iax.add_patch(plt.Circle((0, 0), 1.2, fc="w", ec="0.3", alpha=0.9))
    iax.text(0, 1.02, "N", ha="center", va="bottom", fontsize=7, color="0.3")
    for a_ in np.linspace(sh0 + lo, sh0 + hi, 25):                                  # 95% range of S_Hmax (both ends)
        t = np.radians(a_); iax.plot([-np.sin(t), np.sin(t)], [-np.cos(t), np.cos(t)], color="0.4", alpha=0.12, lw=3, solid_capstyle="butt")
    for az, c, inward in ((sh0, "k", True), (sh0 + 90, "C0", False)):
        for s_ in (0, 180):
            t = np.radians(az + s_); tip, tail = (0.25*np.sin(t), 0.25*np.cos(t)), (0.95*np.sin(t), 0.95*np.cos(t))
            iax.annotate("", xy=tip if inward else tail, xytext=tail if inward else tip,
                         arrowprops=dict(arrowstyle="-|>,head_width=0.35,head_length=0.6", lw=2.5, color=c))
    ax.plot([], [], color="k", lw=3, label=f"S$_{{Hmax}}$: {sh0:.0f}°  (95%: {(sh0 + lo) % 180:.0f}–{(sh0 + hi) % 180:.0f}°, grey)")
    ax.plot([], [], color="C0", lw=3, label=f"S$_{{hmin}}$: {(sh0 + 90) % 180:.0f}°")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.04, pad=0.06, shrink=0.7, label="misfit with the stress (°), best nodal plane")
    ax.legend(loc=free[1], fontsize=8, framealpha=0.9); ax.set_title(title, fontsize=10)
    plt.show()

# the 2019 mechanisms, one by one (same order as the input file of the inversion)
m1, m2 = plane_misfits(SI, catalog.strike, catalog.dip, catalog.rake)
catalog["misfit"] = np.minimum(m1, m2)                                       # best nodal plane
catalog["misfit_chosen"] = SI["faults"].slip_deviation.to_numpy()            # plane chosen by the inversion (3.4)
order = [r for r in REGIME_NAME if r in set(catalog.regime)]
tab = catalog.groupby("regime").agg(n=("misfit", "size"), best_plane=("misfit", "median"), chosen_plane=("misfit_chosen", "median")).reindex(order)
tab.index = [f"{r}: {REGIME_NAME[r]}" for r in tab.index]
print("Median misfit (°) by faulting type, on the best nodal plane and on the plane chosen by the inversion:")
display(tab.round(0).astype(int))
print("Median misfit (°) on the best nodal plane, by quality:",
      ",  ".join(f"{q}: {v:.0f}° ({n} mechanisms)" for q, v, n in catalog.groupby("quality").misfit.agg(["median", "size"]).itertuples()))
fig, ax = plt.subplots(figsize=(7, 3.2)); bins = np.arange(0, 95, 5)
ax.hist(catalog.misfit, bins=bins, histtype="step", lw=2, color="C2", label=f"best nodal plane (median {catalog.misfit.median():.0f}°)")
ax.hist(catalog.misfit_chosen, bins=bins, histtype="step", lw=2, color="k", label=f"plane chosen by the inversion (median {catalog.misfit_chosen.median():.0f}°)")
ax.set_xlabel("misfit with the 2019 stress (°)"); ax.set_ylabel("number of mechanisms"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
sh0, (lo, hi) = shmax_range(SI, BOOT); TREND = fault_trend(*map_extent(catalog))
print(f"S_Hmax {sh0:.0f}° (95%: {(sh0 + lo) % 180:.0f}–{(sh0 + hi) % 180:.0f}°), S_hmin {(sh0 + 90) % 180:.0f}°;  "
      f"mean strike of the mapped faults in the map: {TREND:.0f}°")
stress_map(SI, BOOT, catalog, f"2019 Silivri sequence: {len(catalog)} focal mechanisms colored by their misfit with the stress; mapped faults")

### 3.9 Which nodal plane is the fault?

The polarities cannot tell which of the two nodal planes of a mechanism is the fault (3.2). A stress tensor can try, in two ways:

1. **Fit** (Wallace–Bott hypothesis): the fault is the plane on which the observed slip is closer to the direction of the shear stress, the plane with the smaller misfit (the "best nodal plane" of 3.8).
2. **Instability** (Lund & Slunga, 1999; Vavryčuk, 2014): the fault is the plane closer to failure, with the larger Coulomb stress τ + μσ<sub>n</sub> (σ<sub>n</sub>: normal stress, compression negative; μ: friction). This is the criterion StressInverse uses in every iteration (3.2): the "plane chosen by the inversion".

The next cell counts, for the strike-slip events of 2019, how often each criterion picks the nodal plane closer to the strike of the mapped faults (WNW–ESE to E–W) and how often the other one (NNE–SSW to N–S). Then `explain_planes` shows the numbers behind the two criteria for two quality-A events:

| column | meaning |
|---|---|
| angle to σ₁ | angle between the plane and σ₁; for friction μ the planes closest to failure make ½·arctan(1/μ) ≈ 30° with σ₁ (μ = 0.6) |
| normal stress σ<sub>n</sub> | from −1 (the plane is perpendicular to σ₁: fully clamped) to +1 (perpendicular to σ₃: least clamped) |
| shear stress τ | from 0 to 1 (maximum: planes at 45° from σ₁ and σ₃) |
| Coulomb τ + μσ<sub>n</sub> | the larger, the closer to failure: the instability criterion picks the plane with the larger value |
| observed / predicted rake | direction of slip on the plane: observed (mechanism) and predicted (direction of the shear stress) |
| misfit | angle between the observed and the predicted slip: the fit criterion picks the plane with the smaller value |

In [ ]:
# --- helper code for 3.9 (the two nodal planes in the stress of a run): run the cell; the analysis is at the end ---
from obspy.imaging.beachball import aux_plane

def explain_planes(res, sdr):
    """Both nodal planes of a mechanism in the stress of a run: angle to sigma1, normal and shear stress (normalized:
    sigma_n from -1 = sigma1 to +1 = sigma3, compression negative; shear from 0 to 1), Coulomb stress tau + mu*sigma_n,
    rake predicted by the stress and observed rake, misfit."""
    w, V = np.linalg.eigh(res["tau"]); mu = res["friction"]; rows = []
    n, u = fault_vectors(*sdr); s2, d2, _ = aux_plane(*sdr)
    for (st, dp), nn, uu in (((sdr[0], sdr[1]), n, u), ((s2, d2), u, n)):
        if nn[2] > 0: nn, uu = -nn, -uu
        t = res["tau"] @ nn; sn = t @ nn; sh = t - sn*nn; ts = np.linalg.norm(sh)
        half = (w[2] - w[0])/2; sn_n = (sn - (w[0] + w[2])/2)/half; ts_n = ts/half
        es = np.array([np.cos(np.radians(st)), np.sin(np.radians(st)), 0.0]); sd = max(np.sin(np.radians(dp)), 1e-6)
        rake = lambda v: np.degrees(np.arctan2(-v[2]/sd, v @ es))
        rows.append({"plane (strike/dip)": f"{st % 360:.0f}/{dp:.0f}",
                     "angle to σ1 (°)": round(90 - np.degrees(np.arccos(abs(nn @ V[:, 0])))),
                     "normal stress σn": round(sn_n, 2), "shear stress τ": round(ts_n, 2), "Coulomb τ + μσn": round(ts_n + mu*sn_n, 2),
                     "observed rake (°)": round(rake(uu)), "predicted rake (°)": round(rake(sh/ts)),
                     "misfit (°)": round(np.degrees(np.arccos(np.clip(sh @ uu/ts, -1, 1))))})
    display(pd.DataFrame(rows).set_index("plane (strike/dip)"))

def _strike_diff(a, b):
    """Angle (0-90°) between two strikes, whatever their direction."""
    return np.abs((np.asarray(a, float) - b + 90) % 180 - 90)

# the two nodal planes of every 2019 mechanism, and the plane each criterion picks
aux = np.array([aux_plane(r.strike, r.dip, r.rake) for r in catalog.itertuples()])
m1, m2 = plane_misfits(SI, catalog.strike, catalog.dip, catalog.rake)
f = SI["faults"]
chosen_1 = ((np.abs((f.strike - catalog.strike + 180) % 360 - 180) < 1) & (np.abs(f.dip - catalog.dip) < 1)).to_numpy()  # instability
fit_1 = m1 <= m2                                                                                                            # fit
parallel_1 = _strike_diff(catalog.strike, TREND) <= _strike_diff(aux[:, 0], TREND)       # plane 1 is the one closer to the faults
ss = (catalog.regime == "SS").to_numpy()
count = lambda pick: [int(np.sum((pick == parallel_1) & ss)), int(np.sum((pick != parallel_1) & ss))]
print(f"{ss.sum()} strike-slip events of 2019. Mean strike of the mapped faults in the map: {TREND:.0f}°. Which plane is the fault?")
display(pd.DataFrame({"fit (smaller misfit)": count(fit_1), "instability (StressInverse)": count(chosen_1)},
                     index=["plane closer to the strike of the faults (WNW–ESE)", "the other plane (NNE–SSW)"]).T)
print(f"The two criteria pick the same plane in {np.sum((fit_1 == chosen_1) & ss)} of the {ss.sum()} strike-slip events "
      f"({np.sum(fit_1 == chosen_1)} of all {len(catalog)} mechanisms). "
      f"The misfits of the two planes differ by less than 10° in {np.sum((np.abs(m1 - m2) < 10) & ss)} strike-slip events.")
sig = SI["axes"]
print(f"Stress of the run: σ1 {sig[0][0]:.0f}/{sig[0][1]:.0f}, σ2 {sig[1][0]:.0f}/{sig[1][1]:.0f}, σ3 {sig[2][0]:.0f}/{sig[2][1]:.0f} "
      f"(azimuth/plunge), friction μ = {SI['friction']:.2f}")

# two quality-A strike-slip events: the largest one, and one where both criteria agree
for ev in ("20190926112636", "20190927025542"):
    e = catalog[catalog.event_id.astype(str) == ev].iloc[0]
    print(f"\nEvent {ev} (M {e.magnitude}, quality {e.quality}, nodal-plane uncertainty ≈ {e.fault_plane_uncertainty:.0f}°): "
          f"{e.strike:.0f}/{e.dip:.0f}/{e.rake:.0f}")
    explain_planes(SI, (e.strike, e.dip, e.rake))

**Reading the two tables.** Event 20190926112636 (M 4.4, the largest strike-slip event of quality A):

- **Angle to σ₁.** Both planes make 41–45° with σ₁. Neither is at the optimal angle (≈ 30° for μ = 0.6), and they differ little: this is typical of a strike-slip mechanism in a strike-slip stress field, where the two nodal planes lie almost symmetrically about σ₁.
- **Instability.** The WNW–ESE plane (286/54) is less clamped (σ<sub>n</sub> −0.16 instead of −0.23) and carries more shear stress (0.88 instead of 0.71). Its Coulomb stress is larger (0.78 vs 0.57): it is closer to failure, and StressInverse takes it as the fault.
- **Fit.** But on that plane the stress would push the slip right-laterally with a **normal** component (predicted rake ≈ −156°), while the earthquake slipped right-laterally with a small **reverse** component (rake 167°): misfit 36°. On the NNE–SSW plane (24/80) the predicted and the observed rakes agree (41° and 37°): misfit 4°.

**Why does the direction of slip on the WNW–ESE planes fit so badly?** σ₂ is not vertical: it plunges ≈ 56° towards ≈ 290°, so it lies almost within the steep WNW–ESE planes. On a plane that contains σ₂, the shear stress is perpendicular to σ₂, so this stress pushes the WNW–ESE planes **obliquely**: right-laterally and with the **north side down**, ≈ 30° from the horizontal (predicted rake ≈ −150° if the plane dips north, ≈ +150° if it dips south). Many WNW–ESE planes of 2019 moved with the north side **up** (like 286/54 above): on them the misfit is mostly 25–75°, and the fit criterion prefers the NNE–SSW plane. Where the north side went down (event 20190927025542, second table: 102/74, dipping south, reverse component), the WNW–ESE plane fits well (≈ 9°) and both criteria choose it.

**So which plane is the fault?** For most strike-slip events of 2019 the stress cannot tell:

- the two criteria disagree for about half of them, and for about a third the misfits of the two planes differ by less than 10°, less than the uncertainty of the nodal planes (median ≈ 20° for quality A, 25–30° for B and C: column `fault_plane_uncertainty`);
- what decides the fit is the small dip-slip component of a nearly pure strike-slip mechanism (5–35°), one of the parts of a mechanism that first-motion polarities constrain worst: most rays leave the source close to horizontal (Part 2);
- the 2019 sequence mixes thrust, normal and strike-slip faulting around a blind thrust (3.1): a single uniform stress may not describe all of it, so the "best" plane in this stress is not necessarily the fault.

The fault plane needs independent information: the mapped faults, the alignment of the relocated earthquakes, or the rupture directivity of the larger events. For the stress inversion the choice matters less than one might fear, because both nodal planes share the same P and T axes: the synthetic test of 3.6 showed how often the inversion picks the wrong plane and how well it still recovers the stress.

#### Exercise 10 — Discussion

- Which inputs change the **orientation** of the stress axes, and which ones only change their **uncertainty**?
- The 2019 mechanisms come from a small volume around one rupture. What could go wrong if you mixed earthquakes from fault segments with different stress (e.g. the whole Sea of Marmara)?
- Are the uncertainties printed by StressInverse "true" uncertainties? What do they depend on?
- With how many mechanisms does the result become unstable? What happens to R?
- Most of the catalog are aftershocks of the Mw 5.8 mainshock. The stress inversion assumes a uniform stress that the earthquakes do not change. Could the mainshock have changed the stress around it? How would you test it with this catalog?
- Which mechanisms fit the stress worst (3.8)? Are they of low quality, of one faulting type, or in one place? What could explain them: the wrong nodal plane, errors in the mechanism, or a stress that changes in space or time?
- In 3.9 the fit and the instability criteria often choose different nodal planes. Which one would you trust to identify the fault, and what independent information would you use?
- Would it make sense to test the 2025 mechanisms of Part 2 against this stress? Think of: the 2019 mainshock ruptured a blind thrust with a complex source (Karabulut et al., 2020; Turhan et al., 2023), the 2025 mainshock a different, although related, fault; the two sequences barely overlap; and the 2019 earthquake changed the stress around it and promoted failure in the 2025 area (Karabulut et al., 2020; Şahin et al., 2022). If the 2025 mechanisms did not fit, which of these differences would be responsible? How would you estimate the stress that drove the 2025 sequence instead?
- Wollin et al. (2019) inverted the seismicity of the whole Sea of Marmara region with the same method. Why could a local stress, like the one of the 2019 sequence, differ from the regional one?
- Summarize the stress field of the area in one sentence (regime, σ₁ direction, uncertainty, and how you know it is reliable) as you would write it in a paper.

## References

- Bott, M. H. P. (1959). The mechanics of oblique slip faulting. *Geol. Mag.*, 96, 109–117.
- Durand, V., Bentz, S., Kwiatek, G., Dresen, G., Wollin, C., Heidbach, O., et al. (2020). A two-scale preparation phase preceded an Mw 5.8 earthquake in the Sea of Marmara offshore Istanbul, Turkey. *Seismol. Res. Lett.*, 91, 3139–3147. https://doi.org/10.1785/0220200110
- Hardebeck, J. L., & Shearer, P. M. (2002). A new method for determining first-motion focal mechanisms. *Bull. Seismol. Soc. Am.*, 92, 2264–2276.
- Kagan, Y. Y. (1991). 3-D rotation of double-couple earthquake sources. *Geophys. J. Int.*, 106, 709–716.
- Karabulut, H., et al. (2011). Evolution of the seismicity in the eastern Marmara Sea a decade before and after the 17 August 1999 Izmit earthquake. *Tectonophysics*, 510, 17–27.
- Karabulut, H., Güvercin, S. E., Eskiköy, F., Konca, A. Ö., & Ergintav, S. (2020). The moderate size 2019 September Mw 5.8 Silivri earthquake unveils the complexity of the Main Marmara Fault shear zone. *Geophys. J. Int.*, 224, 377–388. https://doi.org/10.1093/gji/ggaa469
- Lund, B., & Slunga, R. (1999). Stress tensor inversion using detailed microearthquake information and stability constraints: Application to Ölfus in southwest Iceland. *J. Geophys. Res.*, 104, 14947–14964.
- Lund, B., & Townend, J. (2007). Calculating horizontal stress orientations with full or partial knowledge of the tri-axial stress tensor. *Geophys. J. Int.*, 170, 1328–1335. https://doi.org/10.1111/j.1365-246X.2007.03468.x
- Maeda, N. (1985). A method for reading and checking phase times in autoprocessing system of seismic wave data. *Zisin*, 38, 365–379.
- Martínez-Garzón, P., Ben-Zion, Y., Abolfathian, N., Kwiatek, G., & Bohnhoff, M. (2016). A refined methodology for stress inversions of earthquake focal mechanisms. *J. Geophys. Res. Solid Earth*, 121. https://doi.org/10.1002/2016JB013493
- McKenzie, D. P. (1969). The relation between fault plane solutions for earthquakes and the directions of the principal stresses. *Bull. Seismol. Soc. Am.*, 59, 591–601.
- Michael, A. J. (1984). Determination of stress from slip data: Faults and folds. *J. Geophys. Res.*, 89, 11517–11526.
- Núñez-Jara, S., Vera, F., Scotto di Uccio, F., Fabisch, O., Dresen, G., Brantut, N., Bohnhoff, M., & Martínez-Garzón, P. (2026). Enhanced earthquake catalog, focal mechanism catalog, and strong-motion back-projection results for the 2025 Mw 6.3 Central Marmara, Türkiye, earthquake sequence, including relocated events from the 2019 Mw 5.8 Silivri sequence. *GFZ Data Services*. https://doi.org/10.5880/GFZ.ITSJ.2026.002
- Şahin, M., Yaltırak, C., Bulut, F., & Garagon, A. (2022). Stress change generated by the 2019 İstanbul–Silivri earthquakes along the complex structure of the North Anatolian Fault in the Marmara Sea. *Earth Planets Space*, 74, 167. https://doi.org/10.1186/s40623-022-01706-2
- Skoumal, R. J., Hardebeck, J. L., & Shearer, P. M. (2024). SKHASH: A Python package for computing earthquake focal mechanisms. *Seismol. Res. Lett.*, 95, 2519–2526.
- Turhan, F., Acarel, D., Plicka, V., Bohnhoff, M., Polat, R., & Zahradník, J. (2023). Coseismic faulting complexity of the 2019 Mw 5.7 Silivri earthquake in the Central Marmara Seismic Gap, offshore Istanbul. *Seismol. Res. Lett.*, 94, 75–86. https://doi.org/10.1785/0220220111
- Vavryčuk, V. (2014). Iterative joint inversion for stress and fault orientations from focal mechanisms. *Geophys. J. Int.*, 199, 69–77.
- Wallace, R. E. (1951). Geometry of shearing stress and relation to faulting. *J. Geol.*, 59, 118–130.
- Wollin, C., Bohnhoff, M., Vavryčuk, V., & Martínez-Garzón, P. (2019). Stress inversion of regional seismicity in the Sea of Marmara region, Turkey. *Pure Appl. Geophys.*, 176, 1269–1291.
- Zhu, W., & Beroza, G. C. (2019). PhaseNet: a deep-neural-network-based seismic arrival-time picking method. *Geophys. J. Int.*, 216, 261–273.
- Zoback, M. L. (1992). First- and second-order patterns of stress in the lithosphere: The World Stress Map project. *J. Geophys. Res.*, 97, 11703–11728.